<a href="https://colab.research.google.com/github/corbinwhcurtin/smart-finance-assistant/blob/main/starter_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏦 Project Overview

Welcome to your **Smart Finance Assistant** development journey! This notebook will evolve from basic CSV processing to a complete AI-powered finance application.

**Final Application Components:**
- 💬 **AI Chat Interface** - Financial advice personality
- 📊 **Data Analysis** - CSV transaction processing  
- 🔍 **RAG System** - Retrieval from financial documents
- 🛠️ **Custom Tools** - Calculators and utilities
- 🌐 **Gradio UI** - Professional web interface

**Development Approach**: Build progressively from foundation to advanced features, using AI collaboration throughout.

---

# 🚀 Getting Started: Foundation Setup

## Initial Setup
This cell installs the necessary libraries. In a Colab environment, you would uncomment the first line.

In [37]:
# ── Cell 1: Setup & Imports ───────────────────────────────────
!pip install gradio pandas hands-on-ai plotly -q

import os
import json
import calendar
import warnings
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import plotly
import gradio as gr

from datetime import datetime, date
from pathlib import Path
from typing import Optional

warnings.filterwarnings('ignore')

print("Libraries loaded successfully.")
print(f"  Pandas:  {pd.__version__}")
print(f"  Plotly:  {plotly.__version__}")
print(f"  Gradio:  {gr.__version__}")

Libraries loaded successfully.
  Pandas:  2.2.2
  Plotly:  5.24.1
  Gradio:  5.50.0


## Hands-on-AI Configuration

Set up the hands-on-ai package for advanced features (chat, RAG, tools):

In [38]:
# ── Cell 2: AI Configuration ──────────────────────────────────
from hands_on_ai.chat import get_response
from hands_on_ai import agent, rag

import os
os.environ['HANDS_ON_AI_SERVER']  = 'https://ollama.locollm.org'
os.environ['HANDS_ON_AI_MODEL']   = 'gemma3:4b'
os.environ['HANDS_ON_AI_API_KEY'] = 'Curtin2026ISYS20015002'

# Test connection
try:
    test = get_response("Reply with exactly three words: connection is working")
    print("AI connection confirmed.")
    print(f"  Server:   {os.environ['HANDS_ON_AI_SERVER']}")
    print(f"  Model:    {os.environ['HANDS_ON_AI_MODEL']}")
    print(f"  Response: {test}")
except Exception as e:
    print(f"Connection failed: {e}")
    print("Check your network connection and API key.")

AI connection confirmed.
  Server:   https://ollama.locollm.org
  Model:    gemma3:4b
  Response: Connection is working


In [39]:
#Cell 3 Data Persistence (Save / Load Function)

SAVE_FILE = "finance_data.json"

def save_app_data(app_state: dict) -> None:
    """
    Serialises and saves the full app state to a JSON file.
    Called automatically whenever data changes.
    """
    try:
        serialisable = {
            "budget_plan":      app_state.get("budget_plan"),
            "safety_net":       app_state.get("safety_net"),
            "savings_goals":    app_state.get("savings_goals", []),
            "monthly_snapshots": []
        }

        # serialise each monthly snapshot
        for snapshot in app_state.get("monthly_snapshots", []):
            s = snapshot.copy()
            # convert category_summary DataFrame to dict
            if isinstance(s.get("category_summary"), pd.DataFrame):
                s["category_summary"] = s["category_summary"].to_dict()
            serialisable["monthly_snapshots"].append(s)

        with open(SAVE_FILE, "w") as f:
            json.dump(serialisable, f, indent=2, default=str)

    except Exception as e:
        print(f"Auto-save failed: {e}")


def load_app_data() -> dict:
    """
    Loads saved app state from JSON file if it exists.
    Returns empty state if no save file found.
    """
    empty_state = {
        "budget_plan":       None,
        "safety_net":        None,
        "savings_goals":     [],
        "monthly_snapshots": []
    }

    if not Path(SAVE_FILE).exists():
        print("No save file found. Starting fresh.")
        return empty_state

    try:
        with open(SAVE_FILE, "r") as f:
            data = json.load(f)

        # restore category_summary DataFrames
        for snapshot in data.get("monthly_snapshots", []):
            if isinstance(snapshot.get("category_summary"), dict):
                snapshot["category_summary"] = pd.DataFrame.from_dict(
                    snapshot["category_summary"]
                )

        print(f"Save file loaded. {len(data.get('monthly_snapshots', []))} monthly snapshot(s) found.")
        return data

    except Exception as e:
        print(f"Could not load save file: {e}")
        return empty_state


def get_snapshot_label(df: pd.DataFrame) -> str:
    """
    Generates a month/year label from a transaction DataFrame.
    e.g. 'March 2025'
    """
    if df is None or df.empty:
        return "Unknown"
    earliest = df['date'].min()
    latest   = df['date'].max()
    if earliest.month == latest.month:
        return earliest.strftime("%B %Y")
    return f"{earliest.strftime('%b')} – {latest.strftime('%b %Y')}"


# ── Initialise global app state ───────────────────────────────
app_state = load_app_data()
app_state["current_df"]       = None   # active transaction DataFrame
app_state["current_analysis"] = None   # active analysis dictionary
app_state["current_label"]    = None   # e.g. "March 2025"

print(f"App state initialised.")
print(f"  Budget plan:   {'Set' if app_state['budget_plan'] else 'Not set'}")
print(f"  Safety net:    {'Set' if app_state['safety_net'] else 'Not set'}")
print(f"  Savings goals: {len(app_state['savings_goals'])}")
print(f"  Past months:   {len(app_state['monthly_snapshots'])}")

Save file loaded. 1 monthly snapshot(s) found.
App state initialised.
  Budget plan:   Set
  Safety net:    Set
  Savings goals: 1
  Past months:   1


# 🏗️ Foundation: Data Processing Skills

Before building advanced features, establish solid data processing foundations. This section focuses on CSV transaction analysis - the core of your finance assistant.

## Foundation Skill Checkpoint ✅

**Master these basics before advancing to chat/RAG/tools:**
- [ ] Load and clean CSV transaction data
- [ ] Handle real-world data issues (dollar signs, missing values)
- [ ] Calculate spending summaries by category  
- [ ] Generate business-appropriate insights
- [ ] Format output for professional presentation
- [ ] Test functions with various data scenarios

::: {.callout-tip}
## 🤖 AI Collaboration Strategy

For this foundation work, use AI to:
1. **Generate initial code** with specific business context
2. **Handle data cleaning** and validation
3. **Create professional formatting** for outputs
4. **Suggest business insights** from data patterns
5. **Help with testing** edge cases and error handling

**Remember**: You're directing AI like a junior developer - always review and improve their suggestions!
:::

## Sample Transaction Data Setup

Create or load sample transaction data to work with:

---

# 📊 Six-Step Development Methodology

Your notebook must demonstrate the six-step methodology with clear evidence of AI collaboration at each step.

## STEP 1: Understand the Problem

**🎯 Define Your Finance Problem**

In this section, clearly state your chosen finance problem in business terms.

::: {.callout-note}
## Problem Definition Template


I want to help university students understand and alalyse their spending habits from their bank transations (CSV). Focus should be on reducing unnessisary spending during semester but budget for saving and social spending money. It should also provide feedback and insights into were money can "disapear"


AI Prompt

Help me brainstorm features than a csv analysis tool that is useful for university students budgeting while allowing social spending and savings focusing on reducing money that disapears

AI Reply of Features:
1. Catching invisible spending
2. Making a budget
3. Allowing social spending
4. Spending pattern insight
5. Motivation and goal tracking

**Your Problem Statement:**

University Student often struggle budgeting while studying full-time. The goal of the problem is to set budget and analyse spending with a focus of sticking to a budget while still allowing social spending. Using CSV bank transaction data to reduce small transaction that are unnecessary and add up. The program should help students understand their spending habits and provide realistic budget while saving money at the same time.

## STEP 2: Identify Inputs and Outputs

**📥 Define Your Data Flow**

Data inputs:
-	Bank Transaction CSV files with date, description, amount, description
-	Manually enter cash transactions
-	Manually overside incorrect transaction category
-	Monthly budgets, saving goals amount

Processing
-	Clean data to match format (e.g. missing $)
-	Auto categorise transactions
-	Detect reoccurring payments e.g. coffee every day
-	Separate essentials vs other spending e.g. rent vs pub

Insights
-	Essential vs choice spending
-	Invisible spending e.g. small daily payments that can be reduced
-	Subscriptions
-	Daily/weekly allowable spending
-	Social spending amount (is it within reason/% of budget)
-	Where to cut spending
-	Day of week/ payday spending spikes
-	Savings goal progress
-	Financial health score e.g. safety savings bucket
-	AI generated spending avoid in easy to understand English


## Input/Output Analysis Template

Inputs:
CSV file with transaction data (columns: Date, Description, Amount, and optionally Category)
Manually entered transactions (date, description, amount, category) for cash and non-bank spending
Essential flags — user-marked transactions or merchants that cannot be reduced (rent, power, internet)
Category corrections — user edits to fix auto-categorisation errors
Monthly discretionary budget (user-defined)
Savings goal (target amount + target date e.g. "Bali trip — $2,000 by December")
Time period for analysis (e.g. last 30 days, last 3 months)

Outputs:
Essential vs discretionary spend breakdown — shows committed costs before any choices are made
Spending summary by category with totals, averages, and transaction counts
Invisible spend report — small frequent transactions (e.g. coffees, snacks) aggregated and flagged
Subscription audit — recurring charges listed with estimated monthly and annual cost
Day-of-week and post-payday spike analysis — when overspending tends to happen
Social spending health check — social budget as a percentage of discretionary spend, framed positively
Daily discretionary allowance — how much remains per day for the rest of the month
"Cut X → save Y" statements — trade-off insights targeting only reducible spend
Savings goal progress tracker — current progress, weekly contribution needed, projected completion date
Financial health score (0–100) — based on discretionary budget adherence and savings rate
AI-generated personalised advice summary via get_response() — plain-English recommendations grounded in the user's actual reducible spending



## STEP 3: Work the Problem by Hand

**✋ Manual Calculation Examples**

Show 2-3 worked examples to understand the logic before coding.


## Example Business Calculation

Given this sample data:

| Date | Amount | Category | Description |
|------|--------|----------|-------------|
| 2024-08-01 | $45.50 | Groceries | Woolworths |
| 2024-08-02 | $12.00 | Transport | Opal Card |
| 2024-08-03 | $89.95 | Entertainment | Concert |
| 2024-08-04 | -$15.00 | Refund | Returned item |

**Manual Calculations:**
- Total Spending: $45.50 + $12.00 + $89.95 - $15.00 = $132.45
- By Category: Groceries $45.50, Transport $12.00, Entertainment $89.95
- Average Transaction: $132.45 ÷ 4 = $33.11
- Insight: Entertainment represents 67% of positive spending

**🤖 AI Prompt to Try:**


**Your Manual Examples:**
Scenario 1 — Normal weekly spending (typical uni student, Perth)
Date, Description, Amount, Category
2024-03-01, Coles Supermarket Karrinyup, -87.43, Food & Groceries
2024-03-02, Transperth Smartrider Topup, -20.00, Transport
2024-03-03, Boost Juice Hay St, -8.50, Coffee & Snacks
2024-03-04, Spotify Premium, -11.99, Subscriptions
2024-03-04, McDonald's Northbridge, -13.20, Eating Out
2024-03-05, Instagram Boost Meta, -15.00, Other
2024-03-06, Hungry Jacks Murray St, -11.40, Eating Out
2024-03-07, 7-Eleven Beaufort St, -6.30, Coffee & Snacks
2024-03-07, Casual Work Payment, +280.00, Income

Total spending: 87.43 + 20 + 8.50 + 11.99 + 13.20 + 15.00 + 11.40 + 6.30 = 173.82
Total Income: $280.00
By Category: Food & Groceries $87.43, Transport $20.00, Coffee & Snacks $14.80, Subscriptions $11.99, Eating Out $24.60, Other $15.00
Average Transaction (expenses only): $173.82 ÷ 8 = $21.73
Insight: Eating Out ($24.60) and Coffee & Snacks ($14.80) combined represent 22.7% of spending despite being small individual purchases

## STEP 4: Write Pseudocode

**📝 Plan Your Solution Logic**

Sketch the algorithm in plain English before coding.


**Your Pseudocode:**
Smart Finance Assistant — Pseudocode

FUNCTION: Load and Clean Data

Ask the user to upload their bank CSV file
Check the file has at least a date, description, and amount column
Tidy up the data by fixing date formats, removing dollar signs, and deleting any blank or duplicate rows
If the user has added any cash transactions manually, add those into the same list
Hand the cleaned list of transactions to the next step


FUNCTION: Categorise Transactions

Look at the description of each transaction
Compare it against a list of known merchant names and keywords
Assign the most appropriate category such as Food, Transport, or Social
If the transaction already has a category, leave it as is
If nothing matches, label it as Other
Hand the categorised list to the next step


FUNCTION: Flag Essential Transactions

Automatically mark known essential merchants such as rent, power, and phone bills as essential
Show the user a table of all transactions so they can review the categories and essential flags
Allow the user to correct any category that was wrong
Allow the user to mark or unmark any transaction as essential
Whenever the user makes a change, update all the analysis immediately


FUNCTION: Analyse Spending

Separate transactions into three buckets — income, essential expenses, and discretionary expenses
Calculate the total for each bucket
Subtract any refunds from the relevant categories so they are not double counted
For each discretionary category work out the total spent, number of transactions, and average transaction size
Work out how much of the monthly budget has been used and how much is left per day for the rest of the month
Look for invisible spending by finding small purchases under fifteen dollars that happen repeatedly in the same week
Look for subscriptions by finding the same merchant charging a similar amount every month
Return a summary of all these figures


FUNCTION: Detect Spending Patterns

Work out which day of the week the student tends to spend the most
Check if spending jumps noticeably in the days immediately after payday
Look at social spending as a percentage of total discretionary spend and give it a positive label such as healthy balance or worth keeping an eye on
Return these pattern observations


FUNCTION: Track Savings Goal

Take the goal name, target amount, and target date from the user
Work out how much has been saved so far based on income minus all expenses
Calculate what percentage of the goal has been reached
Work out how much needs to be set aside each week to hit the goal on time
For each discretionary category suggest how many weeks sooner the goal could be reached if that category was reduced by half
Flag the goal as at risk if the required weekly saving is unrealistically high
Return the goal progress and suggestions


FUNCTION: Calculate Financial Health Score

Start with a perfect score of 100
Reduce the score if spending has gone over the monthly budget
Reduce the score if invisible spending makes up a large portion of discretionary spend
Reduce the score if subscriptions are taking up too much of the budget
Reduce the score if very little is being saved relative to income
Reduce the score slightly if a payday spending spike was detected
Add points back if the savings goal is on track
Keep the final score between 0 and 100
Assign a plain English label such as On Track, Needs Attention, At Risk, or Action Required
Return the score and label


FUNCTION: Generate AI Advice

Gather a plain English summary of the student's finances including their top spending categories, invisible spend total, subscription costs, savings progress, and health score
Send this summary to the AI along with instructions to be encouraging and non-judgmental
Tell the AI to never suggest cutting essential spending and to always protect a reasonable social budget
Tell the AI to focus first on invisible spend and unused subscriptions
Display the AI's personalised recommendations to the user
Allow the user to ask follow up questions in a chat interface


FUNCTION: Build the Interface

Create a simple four tab interface using Gradio
Tab one lets the user upload their CSV, add manual cash transactions, and edit the transaction table
Tab two shows a spending dashboard with category totals, charts, invisible spend, and subscription alerts
Tab three shows the budget tracker, daily allowance, savings goal progress, and cut X save Y suggestions
Tab four shows the financial health score and the AI advice chatbot
Whenever any data changes in any tab, recalculate everything and refresh all the displays


MAIN PROGRAM

Connect to the AI server
Launch the Gradio interface
Wait for the user to upload their data and interact with the tool
Route each user action to the correct function
Display the results

## STEP 5: Convert to Python

**💻 Implementation with AI Collaboration**

Now implement your solution using AI assistance. Focus on creating professional, business-appropriate code.


## 🤖 Implementation Strategy

**Effective AI Prompts for Implementation:**
```
"I'm implementing a Smart Finance Assistant. Based on my pseudocode, please create
a Python function that [specific functionality]. The code should:
- Handle real-world CSV data issues (dollar signs, missing values)
- Include clear comments explaining business logic
- Use professional variable names
- Format output for business presentation
- Include basic error handling"
```

**Remember to critique and improve AI responses before using them!**
:::

### Foundation Data Processing Functions

In [40]:
# ── Cell 4: Data Loading, Cleaning & Categorisation ──────────

CATEGORIES = [
    "Food & Groceries", "Coffee & Snacks", "Eating Out",
    "Transport", "Social", "Subscriptions", "Shopping",
    "Bills & Utilities", "Other", "Income"
]

KEYWORD_MAP = {
    "Food & Groceries":  ["coles", "woolworths", "aldi", "iga", "foodland",
                          "spudshed", "fresh provisions", "market", "deli"],
    "Coffee & Snacks":   ["boost juice", "7-eleven", "starbucks", "gloria jeans",
                          "cafe", "coffee", "muffin break", "chatime", "gong cha"],
    "Eating Out":        ["mcdonald", "hungry jacks", "kfc", "subway", "nando",
                          "domino", "pizza", "uber eats", "doordash", "menulog",
                          "grill'd", "zambreros", "oporto", "red rooster"],
    "Transport":         ["transperth", "smartrider", "uber", "ola", "didi",
                          "ampol", "bp", "caltex", "shell", "puma energy",
                          "parking", "curtin parking"],
    "Subscriptions":     ["spotify", "netflix", "disney", "youtube", "apple",
                          "amazon prime", "binge", "stan", "paramount",
                          "adobe", "microsoft", "google one"],
    "Bills & Utilities": ["rent", "realmark", "ray white", "lj hooker",
                          "property", "synergy", "alinta", "atco", "water",
                          "optus", "telstra", "vodafone", "tpg", "iinet",
                          "insurance", "medicare", "nib", "bupa"],
    "Shopping":          ["kmart", "target", "big w", "jb hi-fi", "harvey norman",
                          "cotton on", "asos", "uniqlo", "h&m", "myer", "david jones",
                          "ebay", "amazon", "etsy", "chemist warehouse"],
    "Social":            ["hotel", "bar", "pub", "tavern", "nightclub", "club",
                          "grosvenor", "rechabite", "amplifier", "rosie",
                          "lucky chan", "the bird", "fomo", "concert", "ticketek",
                          "eventbrite", "bowling", "hoyts", "event cinemas"],
    "Income":            ["salary", "payroll", "wage", "casual work", "pay",
                          "centrelink", "youth allowance", "austudy",
                          "scholarship", "transfer in", "deposit"]
}

ESSENTIAL_KEYWORDS = [
    "rent", "realmark", "ray white", "lj hooker", "property",
    "synergy", "alinta", "atco", "water corporation",
    "optus", "telstra", "vodafone", "tpg", "iinet",
    "insurance", "medicare", "hospital"
]


def load_and_clean_transaction_data(file_path):
    """
    Loads and cleans bank transaction CSV data.
    Handles Australian bank export formats.

    Parameters:
        file_path: path string, Path object, or existing DataFrame
    Returns:
        Cleaned DataFrame or None if loading fails
    """

    # ── Step 1: Load ──────────────────────────────────────────
    try:
        if isinstance(file_path, pd.DataFrame):
            df = file_path.copy()
        else:
            df = pd.read_csv(file_path)
        print(f"Loaded {len(df)} rows.")
    except FileNotFoundError:
        print("File not found. Check the file path and try again.")
        return None
    except Exception as e:
        print(f"Could not read file: {e}")
        return None

    # ── Step 2: Standardise column names ─────────────────────
    df.columns = df.columns.str.strip().str.lower()
    rename_map = {
        "transaction date": "date", "trans date": "date",
        "debit":            "amount", "credit": "amount",
        "merchant":         "description", "details": "description",
        "type":             "category", "memo": "description"
    }
    df = df.rename(columns={
        c: rename_map[c] for c in df.columns if c in rename_map
    })

    # ── Step 3: Validate required columns ────────────────────
    required = ["date", "amount", "description"]
    missing  = [c for c in required if c not in df.columns]
    if missing:
        print(f"Missing required columns: {missing}")
        print(f"Your file has: {list(df.columns)}")
        return None

    # ── Step 4: Clean Amount ──────────────────────────────────
    if df["amount"].dtype == object:
        df["amount"] = (df["amount"]
                        .astype(str)
                        .str.replace("$", "", regex=False)
                        .str.replace(",", "", regex=False)
                        .str.replace("+", "", regex=False)
                        .str.strip())
    df["amount"] = pd.to_numeric(df["amount"], errors="coerce")
    invalid = df["amount"].isna().sum()
    if invalid:
        print(f"  {invalid} rows removed — unreadable amounts.")
    df = df.dropna(subset=["amount"])
    df = df[df["amount"] != 0]

    # ── Step 5: Clean Date ────────────────────────────────────
    df["date"] = pd.to_datetime(df["date"], dayfirst=True, errors="coerce")
    invalid = df["date"].isna().sum()
    if invalid:
        print(f"  {invalid} rows removed — unreadable dates.")
    df = df.dropna(subset=["date"])

    # ── Step 6: Clean Description ─────────────────────────────
    df["description"] = (df["description"]
                         .astype(str)
                         .str.strip()
                         .str.upper())
    df = df[~df["description"].isin(["", "NAN"])]

    # ── Step 7: Add missing columns ──────────────────────────
    if "category" not in df.columns:
        df["category"] = "Uncategorised"
    if "essential" not in df.columns:
        df["essential"] = False

    # ── Step 8: Remove duplicates ─────────────────────────────
    before = len(df)
    df = df.drop_duplicates(subset=["date", "description", "amount"])
    removed = before - len(df)
    if removed:
        print(f"  {removed} duplicate rows removed.")

    # ── Step 9: Sort ──────────────────────────────────────────
    df = df.sort_values("date").reset_index(drop=True)

    date_range = (f"{df['date'].min().strftime('%d %b %Y')} "
                  f"to {df['date'].max().strftime('%d %b %Y')}")
    print(f"  Period:   {date_range}")
    print(f"  Income:   {len(df[df['amount'] > 0])} transactions")
    print(f"  Expenses: {len(df[df['amount'] < 0])} transactions")

    return df


def categorise_transactions(df: pd.DataFrame) -> pd.DataFrame:
    """
    Auto-assigns categories based on keyword matching.
    Preserves existing categories if already set.
    """
    df = df.copy()

    for idx, row in df.iterrows():
        if row["category"] not in ["Uncategorised", "", None]:
            continue
        desc  = str(row["description"]).lower()
        found = False
        for category, keywords in KEYWORD_MAP.items():
            if any(kw in desc for kw in keywords):
                df.at[idx, "category"] = category
                found = True
                break
        if not found:
            # income = positive amount
            if row["amount"] > 0:
                df.at[idx, "category"] = "Income"
            else:
                df.at[idx, "category"] = "Other"

    categorised = (df["category"] != "Uncategorised").sum()
    print(f"Categorisation complete. {categorised}/{len(df)} transactions assigned.")
    return df


def flag_essential_transactions(df: pd.DataFrame) -> pd.DataFrame:
    """
    Auto-flags known essential merchants.
    User can override in the edit step.
    """
    df = df.copy()

    flagged_count = 0
    for idx, row in df.iterrows():
        desc = str(row["description"]).lower()
        if any(kw in desc for kw in ESSENTIAL_KEYWORDS):
            df.at[idx, "essential"] = True
            flagged_count += 1

    print(f"Essential flagging complete. {flagged_count} transactions flagged.")
    return df


# ── Quick test ────────────────────────────────────────────────
print("\nCell 4 ready — data loading and categorisation functions defined.")


Cell 4 ready — data loading and categorisation functions defined.


In [41]:
# ── Cell 5: Spending Analysis Engine ─────────────────────────

def analyse_spending_by_category(df: pd.DataFrame) -> dict:
    """
    Analyses cleaned transaction data by category.
    Calculates totals, percentages, and generates student-focused insights.

    Parameters:
        df: cleaned DataFrame from load_and_clean_transaction_data()
    Returns:
        Dictionary containing full analysis summary
    """

    if df is None or df.empty:
        print("No transaction data found. Please load your CSV first.")
        return None

    # ── Step 1: Separate transaction types ───────────────────
    income_df        = df[df["amount"] > 0].copy()
    essential_df     = df[(df["essential"] == True) & (df["amount"] < 0)].copy()
    discretionary_df = df[(df["essential"] == False) & (df["amount"] < 0)].copy()
    refunds_df       = df[(df["amount"] > 0) &
                          (~df["category"].isin(["Income"]))].copy()

    # ── Step 2: Top level totals ──────────────────────────────
    total_income        = income_df["amount"].sum()
    total_essential     = abs(essential_df["amount"].sum())
    total_refunds       = refunds_df["amount"].sum()
    gross_discretionary = abs(discretionary_df["amount"].sum())
    net_discretionary   = gross_discretionary - total_refunds
    net_position        = total_income - total_essential - net_discretionary
    total_expenses      = total_essential + net_discretionary
    essential_pct       = (total_essential / total_income * 100) if total_income > 0 else 0
    savings_rate        = (net_position / total_income * 100) if total_income > 0 else 0

    # ── Step 3: Category breakdown (all expenses) ────────────
    all_expenses_df  = df[df["amount"] < 0].copy()
    category_summary = (
        all_expenses_df
        .groupby("category")["amount"]
        .agg(["sum", "count", "mean"])
        .rename(columns={"sum": "total", "count": "transactions", "mean": "average"})
    )
    category_summary["total"]   = category_summary["total"].abs()
    category_summary["average"] = category_summary["average"].abs()

    total_all_expenses = category_summary["total"].sum()
    category_summary["percentage"] = (
        category_summary["total"] / total_all_expenses * 100
    ).round(1)

    essential_categories = (
        all_expenses_df[all_expenses_df["essential"] == True]["category"]
        .unique().tolist()
    )
    category_summary["essential"] = category_summary.index.isin(essential_categories)
    category_summary = category_summary.sort_values("total", ascending=False)

    # ── Step 4: Invisible spending ────────────────────────────
    invisible_categories = ["Coffee & Snacks", "Eating Out", "Other"]
    invisible_df    = discretionary_df[
        (discretionary_df["amount"].abs() < 15) &
        (discretionary_df["category"].isin(invisible_categories))
    ]
    invisible_total  = abs(invisible_df["amount"].sum())
    invisible_count  = len(invisible_df)
    invisible_annual = invisible_total * 12

    # ── Step 5: Subscription audit ───────────────────────────
    subscription_df     = discretionary_df[
        discretionary_df["category"] == "Subscriptions"
    ].copy()
    subscription_total  = abs(subscription_df["amount"].sum())
    subscription_annual = subscription_total * 12

    # ── Step 6: Social spending ───────────────────────────────
    social_total = abs(
        discretionary_df[
            discretionary_df["category"] == "Social"]["amount"].sum()
    )
    social_pct = (social_total / net_discretionary * 100) if net_discretionary > 0 else 0

    if social_pct < 5:
        social_label = "Very low — social connection matters for wellbeing"
    elif social_pct <= 20:
        social_label = "Reasonable"
    elif social_pct <= 35:
        social_label = "High — worth reviewing"
    else:
        social_label = "Excessive — this is significantly impacting your budget"

    # ── Step 7: Day of week pattern ───────────────────────────
    discretionary_df = discretionary_df.copy()
    discretionary_df["day_of_week"] = discretionary_df["date"].dt.day_name()
    day_spend = (
        discretionary_df.groupby("day_of_week")["amount"]
        .sum().abs()
        .reindex(["Monday","Tuesday","Wednesday","Thursday",
                  "Friday","Saturday","Sunday"])
        .fillna(0)
    )
    highest_spend_day = day_spend.idxmax()

    # ── Step 8: Financial health score ───────────────────────
    score = 100

    # essential expenses as % of income
    if essential_pct > 60:
        score -= 20
    elif essential_pct > 50:
        score -= 10

    # budget adherence
    if net_position < 0:
        overspend_pct = abs(net_position) / total_income * 100
        score -= min(30, int(overspend_pct))

    # invisible spending
    if invisible_total > 0:
        invisible_pct = invisible_total / net_discretionary * 100
        if invisible_pct > 15:
            score -= 10
        elif invisible_pct > 8:
            score -= 5

    # subscriptions
    if subscription_total > 0:
        sub_pct = subscription_total / net_discretionary * 100
        if sub_pct > 20:
            score -= 10
        elif sub_pct > 10:
            score -= 5

    # savings rate
    if savings_rate < 0:
        score -= 20
    elif savings_rate < 10:
        score -= 10

    # social overspending
    if social_pct > 35:
        score -= 10
    elif social_pct > 25:
        score -= 5

    score = max(0, min(100, score))

    if score >= 80:
        health_label, health_colour = "On Track",        "green"
    elif score >= 60:
        health_label, health_colour = "Needs Attention", "orange"
    elif score >= 40:
        health_label, health_colour = "At Risk",         "red"
    else:
        health_label, health_colour = "Action Required", "darkred"

    # ── Step 9: Generate insights ─────────────────────────────
    insights = []

    if net_position < 0:
        insights.append({
            "severity": "critical",
            "title":    "Spending Exceeds Income",
            "detail":   (
                f"You spent ${abs(net_position):.2f} more than you earned. "
                f"This is not sustainable. Essential expenses alone account for "
                f"{essential_pct:.1f}% of your income (${total_essential:.2f}), "
                f"leaving only ${total_income - total_essential:.2f} for everything else. "
                f"Your discretionary spending of ${net_discretionary:.2f} exceeds that by "
                f"${net_discretionary - (total_income - total_essential):.2f}."
            )
        })
    elif essential_pct > 50:
        insights.append({
            "severity": "warning",
            "title":    "High Fixed Cost Burden",
            "detail":   (
                f"Essential expenses take up {essential_pct:.1f}% of your income. "
                f"This is a significant constraint. You have only "
                f"${total_income - total_essential:.2f} per period for all "
                f"discretionary spending and savings combined."
            )
        })

    if invisible_total > 0:
        insights.append({
            "severity": "warning",
            "title":    "Invisible Spending Detected",
            "detail":   (
                f"${invisible_total:.2f} was spent across {invisible_count} small "
                f"transactions under $15. That is ${invisible_annual:.2f} per year "
                f"on purchases individually too small to notice but collectively "
                f"significant. Coffees, snacks, and convenience store visits "
                f"are the usual culprits."
            )
        })

    if subscription_total > 0:
        sub_pct = subscription_total / net_discretionary * 100
        insights.append({
            "severity": "warning" if sub_pct > 10 else "info",
            "title":    "Subscription Costs",
            "detail":   (
                f"Subscriptions cost ${subscription_total:.2f} per month "
                f"(${subscription_annual:.2f} per year) — {sub_pct:.1f}% of "
                f"discretionary spending. Review each one: if you have not used "
                f"a service at least 4 times this month, cancel it."
            )
        })

    if social_pct > 25:
        insights.append({
            "severity": "warning",
            "title":    "Social Spending is High",
            "detail":   (
                f"Social spending of ${social_total:.2f} represents {social_pct:.1f}% "
                f"of your discretionary budget. Some social spending is healthy and "
                f"necessary, but at this level it is a primary driver of budget pressure. "
                f"Setting a fixed weekly social limit and sticking to it would have "
                f"an immediate impact."
            )
        })

    # top discretionary category
    disc_summary = category_summary[category_summary["essential"] == False]
    if not disc_summary.empty:
        top_cat      = disc_summary.index[0]
        top_cat_row  = disc_summary.iloc[0]
        if top_cat not in ["Social"] or social_pct <= 25:
            insights.append({
                "severity": "info",
                "title":    f"Largest Discretionary Category: {top_cat}",
                "detail":   (
                    f"{top_cat} accounts for ${top_cat_row['total']:.2f} "
                    f"({top_cat_row['percentage']:.1f}% of all expenses). "
                    f"Reducing this category by 25% would save "
                    f"${top_cat_row['total'] * 0.25:.2f} per period."
                )
            })

    insights.append({
        "severity": "info",
        "title":    f"Highest Spending Day: {highest_spend_day}",
        "detail":   (
            f"You tend to spend the most on {highest_spend_day}s. "
            f"Being aware of this pattern can help you pause before "
            f"discretionary purchases on that day."
        )
    })

    # ── Step 10: Print console summary ───────────────────────
    print("=" * 55)
    print("         SMART FINANCE ASSISTANT — SUMMARY")
    print("=" * 55)
    print(f"  Total Income:          ${total_income:>9.2f}")
    print(f"  Essential Expenses:    ${total_essential:>9.2f}  ({essential_pct:.1f}% of income)")
    print(f"  Discretionary Spend:   ${net_discretionary:>9.2f}")
    if net_position < 0:
        print(f"  Net Position:          ${net_position:>9.2f}  ** DEFICIT **")
    else:
        print(f"  Net Position:          ${net_position:>9.2f}")
    print(f"  Health Score:          {score}/100 — {health_label}")
    print(f"{'─' * 55}")
    print(f"  {'Category':<22} {'Total':>8}  {'%':>5}  {'Type':<12}")
    print(f"{'─' * 55}")
    for cat, row in category_summary.iterrows():
        cat_type = "Essential" if row["essential"] else "Flexible"
        print(f"  {cat:<22} ${row['total']:>7.2f}  {row['percentage']:>4.1f}%  {cat_type}")
    print("=" * 55)

    return {
        "total_income":        total_income,
        "total_essential":     total_essential,
        "net_discretionary":   net_discretionary,
        "net_position":        net_position,
        "total_expenses":      total_expenses,
        "essential_pct":       essential_pct,
        "savings_rate":        savings_rate,
        "category_summary":    category_summary,
        "invisible_total":     invisible_total,
        "invisible_count":     invisible_count,
        "invisible_annual":    invisible_annual,
        "subscription_total":  subscription_total,
        "subscription_annual": subscription_annual,
        "subscription_df":     subscription_df,
        "social_total":        social_total,
        "social_pct":          social_pct,
        "social_label":        social_label,
        "day_spend":           day_spend,
        "highest_spend_day":   highest_spend_day,
        "health_score":        score,
        "health_label":        health_label,
        "health_colour":       health_colour,
        "insights":            insights,
    }


print("Cell 5 ready — analysis engine defined.")

Cell 5 ready — analysis engine defined.


In [42]:
# ── Cell 6: RAG Knowledge Base ────────────────────────────────

from hands_on_ai import rag
from pathlib import Path

FINANCE_KNOWLEDGE = """
BUDGETING BASICS
The 50/30/20 rule is a starting framework: 50% of income to needs, 30% to wants, and 20% to savings. For students on irregular income, cover essentials first, set a fixed weekly discretionary limit, and automate savings transfers on payday. Tracking every transaction, even small ones, is the single most effective budgeting habit. A budget only works if it reflects reality — set limits based on what you actually spend, then reduce gradually.

COFFEE AND SNACKS
Coffee and snack purchases are the classic invisible spending trap. A $5 coffee bought 5 days a week costs $1,300 per year. Swapping 3 of those for home brews weekly saves around $780 annually. The issue is not one coffee — it is the habit of daily convenience spending that accumulates without notice.

SUBSCRIPTIONS
Subscription creep is one of the most common budget leaks for students. List every subscription and ask: did I use this at least 4 times last month? Sharing plans with housemates halves costs immediately. A $15 subscription unused for 6 months has cost $90 for nothing. Streaming services, gym memberships, and app subscriptions should be audited every 3 months without exception.

FOOD AND GROCERIES
Switching from Woolworths or Coles to ALDI for staples saves 20 to 30 percent on average. Meal prepping on Sundays eliminates the temptation to order delivery during busy uni weeks. Writing a shopping list and sticking to it prevents impulse purchases. Buying in bulk for non-perishables reduces cost per unit significantly over time.

EATING OUT AND FOOD DELIVERY
Food delivery apps are the single biggest discretionary budget threat for students. A $25 Uber Eats order twice a week totals $2,600 per year. The convenience premium on delivery is typically 30 to 50 percent above cooking at home. Setting a hard limit of two takeaway meals per week and cooking the rest saves $100 to $150 per month without significant lifestyle sacrifice.

SOCIAL SPENDING
Social spending should be budgeted for, not eliminated. Cutting social activities entirely leads to burnout and the abandonment of budgets altogether. Set a fixed social budget per week and treat it as a real limit. Pre-drinks at home, free campus events, and cheaper venue choices still allow a full social life at a fraction of the cost. The goal is controlled social spending, not zero social spending.

EXCESSIVE SOCIAL SPENDING
When social spending exceeds 25 percent of discretionary income it becomes a primary budget problem. Alcohol, Uber rides home, entry fees, and late-night food combine quickly. A single big night out can cost $100 to $150 all-in. Setting a per-night cash limit and leaving the card at home is one of the most effective controls. Choosing venues with no entry fee, drinking less, and splitting Ubers all reduce the per-outing cost significantly.

INVISIBLE SPENDING
Invisible spending is money that leaves the account in amounts too small to notice individually but large enough to matter in total. Convenience store visits, vending machines, small app purchases, and ATM fees are common culprits. Most students find $80 to $150 disappearing this way each month when they total it up. The fix is simple: categorise and total all transactions under $15 once a month and confront the number directly.

SHOPPING AND IMPULSE BUYING
The 48-hour rule is highly effective for non-essential purchases over $50 — wait two days before buying. Unsubscribing from retail email lists removes the trigger. Selling unused items on Facebook Marketplace before buying new things creates a natural friction. Most impulse purchases feel less urgent 48 hours later.

SAVINGS GOALS
Saving is significantly easier when tied to a specific goal with a deadline. Name the goal, set the amount, set the date, and work backwards to a weekly savings target. Automating a transfer to a separate savings account on payday removes the decision entirely. Even $20 per week builds a $1,000 emergency fund in less than a year.

SAFETY NET FUND
A safety net fund is more important than a holiday savings goal. Casual workers have no sick leave — one bad week of illness can mean missing rent. A $500 buffer covers one missed pay. A $1,000 buffer covers two weeks of illness or a car repair. One month of essential expenses as a buffer covers most realistic emergencies. Build the safety net before saving for anything else.

EMERGENCY FUND
Without an emergency fund, any unexpected cost goes on a credit card or wipes out a savings goal. Start with $500, then build to one month of essential expenses. Keep it in a separate account so it is out of sight. Do not touch it for non-emergencies — it is not a holiday fund.

PAYDAY HABITS
Payday splurging is extremely common. The feeling of a full account triggers spending that unravels the entire month's budget within days. Automate transfers on payday in this order: savings first, then bills, then a weekly allowance. Treat the weekly allowance as the only available money, even if more is saved elsewhere.

TRANSPORT
A Transperth concession SmartRider is significantly cheaper than driving and parking in Perth. Uber rides home from nights out add up fast — splitting with friends or pre-booking reduces the cost. Carpooling to uni cuts petrol and parking costs by half. These small transport choices compound over a semester.

INCOME AND IRREGULAR WORK
Base the budget on minimum expected income, not the best week. Casual work income is unreliable — one roster change can cut weekly earnings significantly. Centrelink Youth Allowance and Austudy provide a reliable baseline. Any income above the minimum should go to savings, not lifestyle inflation. Never increase fixed commitments based on irregular income.

ENTERTAINMENT OVERSPENDING
Set a fixed entertainment budget and treat it as a hard limit. When it runs out for the week, stop. Free alternatives include campus events, parks, libraries, and streaming shared with housemates. The problem is not entertainment itself — it is entertainment without a limit.

RENT PRESSURE
When rent exceeds 30 percent of income, financial pressure is significant. At 40 percent or above, there is mathematically very little room for anything else. Rent cannot typically be reduced in the short term but it should be a primary consideration when renewing leases or choosing accommodation. Living closer to university or with more housemates directly reduces this pressure.
"""

def setup_finance_rag():
    """
    Builds the RAG knowledge base from the finance knowledge text.
    Uses keyword-based retrieval since embeddings are not supported
    on the current server.

    Returns:
        callable: ask() function
    """

    # save knowledge to file and chunk it
    knowledge_path = Path("finance_knowledge.txt")
    with open(knowledge_path, "w") as f:
        f.write(FINANCE_KNOWLEDGE)

    text   = rag.load_text_file(knowledge_path)
    chunks = rag.chunk_text(text)
    print(f"Knowledge base ready. {len(chunks)} chunks loaded.")

    def ask(question: str, k: int = 3) -> str:
        """
        Retrieves the most relevant knowledge chunks for a question
        and generates a grounded AI answer.

        Args:
            question: finance question to answer
            k:        number of chunks to retrieve
        Returns:
            str: grounded AI response
        """

        # keyword scoring
        question_words = set(question.lower().split())
        scored = sorted(
            [(len(question_words & set(c.lower().split())), c) for c in chunks],
            key=lambda x: x[0],
            reverse=True
        )
        context = "\n\n".join(chunk for _, chunk in scored[:k])

        prompt = f"""
You are a direct, no-nonsense financial advisor for Australian university students.
Answer using ONLY the retrieved knowledge below.
Be specific, reference dollar amounts where possible, and do not soften bad news.
Keep your answer to 4 sentences maximum.

KNOWLEDGE:
{context}

QUESTION: {question}

ANSWER:"""

        try:
            return get_response(prompt)
        except Exception as e:
            return f"Could not reach AI: {e}"

    return ask


# initialise RAG
finance_rag_ask = setup_finance_rag()


# wrapper object for consistent syntax
class FinanceRAG:
    def ask(self, question):
        return finance_rag_ask(question)

rag_system = FinanceRAG()

print("Cell 6 ready — RAG system initialised.")

Knowledge base ready. 3 chunks loaded.
Cell 6 ready — RAG system initialised.


## Custom Financial Tools

In [43]:
# ── Cell 7: Agent Tool — Savings Calculator ───────────────────

from hands_on_ai import agent

def create_savings_calculator_tool():
    """
    Registers a savings goal calculator as an agent tool.
    Accepts a single string input for hands_on_ai agent compatibility.
    """

    def savings_goal_calculator(input: str) -> str:
        """
        Calculates time to reach a savings goal.
        Input format: "target=2000, monthly=150, current=350"
        """

        # ── Parse input string ────────────────────────────────
        try:
            params = {}
            for part in input.split(","):
                if "=" in part:
                    key, value = part.strip().split("=")
                    params[key.strip()] = float(value.strip())

            target_amount        = params.get("target", 0)
            monthly_contribution = params.get("monthly", 0)
            current_savings      = params.get("current", 0)
        except Exception:
            return (
                "Could not read inputs. "
                "Please provide: target=2000, monthly=150, current=350"
            )

        # ── Validate ──────────────────────────────────────────
        if target_amount <= 0:
            return "Target amount must be greater than $0."
        if monthly_contribution <= 0:
            return "Monthly contribution must be greater than $0."
        if current_savings < 0:
            return "Current savings cannot be negative."
        if current_savings >= target_amount:
            return (
                f"Goal already reached. "
                f"Current savings ${current_savings:.2f} meets "
                f"target of ${target_amount:.2f}."
            )

        # ── Calculate ─────────────────────────────────────────
        amount_remaining    = target_amount - current_savings
        months_to_goal      = amount_remaining / monthly_contribution
        full_months         = int(months_to_goal)
        extra_weeks         = round((months_to_goal - full_months) * 4)
        years               = full_months // 12
        months              = full_months % 12
        weekly_equivalent   = monthly_contribution / 4.33
        progress_pct        = (current_savings / target_amount) * 100

        # ── Build time string ─────────────────────────────────
        if years > 0 and months > 0:
            time_str = (f"{years} year{'s' if years > 1 else ''} "
                        f"and {months} month{'s' if months > 1 else ''}")
        elif years > 0:
            time_str = f"{years} year{'s' if years > 1 else ''}"
        elif full_months > 0 and extra_weeks > 0:
            time_str = (f"{full_months} month{'s' if full_months > 1 else ''} "
                        f"and {extra_weeks} week{'s' if extra_weeks > 1 else ''}")
        elif full_months > 0:
            time_str = f"{full_months} month{'s' if full_months > 1 else ''}"
        else:
            time_str = f"{extra_weeks} week{'s' if extra_weeks > 1 else ''}"

        # ── Format output ─────────────────────────────────────
        return (
            f"SAVINGS GOAL CALCULATOR\n"
            f"{'─' * 38}\n"
            f"Goal:                  ${target_amount:>10.2f}\n"
            f"Already saved:         ${current_savings:>10.2f}\n"
            f"Still needed:          ${amount_remaining:>10.2f}\n"
            f"Progress:              {progress_pct:>9.1f}%\n"
            f"{'─' * 38}\n"
            f"Monthly contribution:  ${monthly_contribution:>10.2f}\n"
            f"Weekly equivalent:     ${weekly_equivalent:>10.2f}\n"
            f"Time to reach goal:    {time_str}\n"
            f"{'─' * 38}"
        )

    # ── Register with agent ───────────────────────────────────
    agent.register_tool(
        name        = "savings_goal_calculator",
        description = (
            "Calculates how long it will take to reach a savings goal. "
            "Input format: 'target=2000, monthly=150, current=350' "
            "where target is the goal amount, monthly is the monthly "
            "contribution, and current is the amount already saved."
        ),
        function    = savings_goal_calculator
    )

    print(f"Savings calculator registered.")
    print(f"  Registered tools: {agent.list_tools()}")
    return savings_goal_calculator


# initialise the tool
savings_calculator = create_savings_calculator_tool()


Savings calculator registered.
  Registered tools: [{'name': 'savings_goal_calculator', 'description': "Calculates how long it will take to reach a savings goal. Input format: 'target=2000, monthly=150, current=350' where target is the goal amount, monthly is the monthly contribution, and current is the amount already saved."}]


In [44]:
# ── Cell 8: AI Recommendations & Finn Chatbot ────────────────

def generate_ai_recommendations(analysis: dict) -> str:
    """
    Generates personalised financial recommendations based on
    spending analysis. Direct, specific, and referenced to
    the student's actual numbers.

    Args:
        analysis: dictionary from analyse_spending_by_category()
    Returns:
        str: formatted recommendations report
    """

    if analysis is None:
        return "No analysis data available. Please load and analyse a CSV first."

    # ── Build category breakdown text ─────────────────────────
    essential_lines      = []
    discretionary_lines  = []
    for cat, row in analysis["category_summary"].iterrows():
        line = (f"  {cat}: ${row['total']:.2f} "
                f"({row['percentage']:.1f}% of total expenses)")
        if row["essential"]:
            essential_lines.append(line)
        else:
            discretionary_lines.append(line)

    essential_text     = "\n".join(essential_lines)     or "  None flagged"
    discretionary_text = "\n".join(discretionary_lines) or "  None flagged"

    # ── Build subscription list ───────────────────────────────
    sub_lines = []
    for _, row in analysis["subscription_df"].iterrows():
        sub_lines.append(
            f"  {row['description']}: "
            f"${abs(row['amount']):.2f}/month "
            f"(${abs(row['amount']) * 12:.2f}/year)"
        )
    sub_text = "\n".join(sub_lines) or "  None detected"

    # ── Build deficit context ─────────────────────────────────
    if analysis["net_position"] < 0:
        position_context = (
            f"CRITICAL: Student is in deficit of ${abs(analysis['net_position']):.2f}. "
            f"They are spending more than they earn. Be direct about this."
        )
    elif analysis["savings_rate"] < 10:
        position_context = (
            f"WARNING: Savings rate is only {analysis['savings_rate']:.1f}%. "
            f"Student is barely breaking even."
        )
    else:
        position_context = (
            f"Student has a positive net position of ${analysis['net_position']:.2f} "
            f"and a savings rate of {analysis['savings_rate']:.1f}%."
        )

    # ── Build prompt ──────────────────────────────────────────
    prompt = f"""
You are a direct, no-nonsense financial advisor for Australian university students.
You are honest but not unreasonable. You understand that uni students have a social life
and that small pleasures like a coffee are acceptable and healthy.

STUDENT FINANCIAL SUMMARY:
- Income this period:          ${analysis['total_income']:.2f}
- Essential expenses (fixed):  ${analysis['total_essential']:.2f} ({analysis['essential_pct']:.1f}% of income)
- Discretionary spending:      ${analysis['net_discretionary']:.2f}
- Net position:                ${analysis['net_position']:.2f}
- Financial health score:      {analysis['health_score']}/100 — {analysis['health_label']}
- {position_context}

ESSENTIAL EXPENSES — do not suggest reducing these:
{essential_text}

DISCRETIONARY EXPENSES — these are the only targets for savings advice:
{discretionary_text}

SUBSCRIPTIONS:
{sub_text}

OTHER FINDINGS:
- Invisible spending (under $15): ${analysis['invisible_total']:.2f} across {analysis['invisible_count']} transactions (${analysis['invisible_annual']:.2f}/year)
- Social spending: ${analysis['social_total']:.2f} ({analysis['social_pct']:.1f}%) — {analysis['social_label']}
- Highest spending day: {analysis['highest_spend_day']}

Provide a financial report with these exact sections:

1. FINANCIAL REALITY CHECK
   Be honest about the student's overall financial position in 2-3 sentences.
   If they are in surplus, acknowledge it. If deficit, state the number clearly.

2. TOP 3 SAVINGS OPPORTUNITIES
   Identify the 3 most impactful changes targeting discretionary spending only.
   PRIORITY ORDER for savings advice:
   - Food delivery apps (Uber Eats, DoorDash) first — these are the biggest value-for-money trap
   - Eating out at fast food chains second
   - Large social nights with high per-outing cost third
   Each must include a specific dollar saving per month if the advice is followed.
   Do not suggest reducing rent, utilities, or phone bills.

3. INVISIBLE SPENDING REALITY
   Small purchases like 1-2 coffees per week are acceptable and not worth stressing over.
   Only flag invisible spending if it is genuinely excessive (over $100/month total).
   If it is moderate, acknowledge it briefly and move on — do not lecture about coffee.
   Focus on habitual convenience store stops and vending machines as the real culprits,
   not the occasional treat.

4. SUBSCRIPTION AUDIT
   List each subscription and give a keep or cancel recommendation with a reason.
   A subscription is worth keeping if it is used regularly and costs under $15/month.
   State the annual saving if cancellable subscriptions are removed.

5. SOCIAL SPENDING ASSESSMENT
   Social spending under 25% of discretionary income is reasonable for a university student.
   Social spending between 25-35% is worth monitoring but not alarming.
   Only flag as excessive if it is above 35% of discretionary income.
   IMPORTANT: Any weekly limit you recommend must be mathematically consistent with
   the monthly total you cite. If you say monthly spending is $X, your weekly limit
   must equal $X divided by 4.33 — never recommend a weekly limit that implies
   the same or higher monthly spend than you just called excessive.
   Acknowledge that socialising is a legitimate and important part of uni life.

6. PRIORITY ACTION THIS WEEK
   One single specific change the student can make before next week.
   Focus on food delivery or eating out as the first target — these have the
   best return on effort and do not require cutting enjoyable social activities.
   Be specific, reference their actual spending.

Tone: direct and honest, but not preachy. Do not lecture about small purchases.
Do not repeat the same point across multiple sections.
Reference the student's real numbers in every section.
Do not suggest reducing essential expenses under any circumstances.
"""

    try:
        response = get_response(prompt)
        return response
    except Exception as e:
        return f"Could not reach AI: {e}"


def build_finn_system_prompt(analysis: dict = None, budget_plan: dict = None) -> str:
    """
    Builds Finn's system prompt with full financial context.
    Finn is direct, honest, and does not soften bad news.

    Args:
        analysis:    current spending analysis dictionary
        budget_plan: current budget plan dictionary
    Returns:
        str: system prompt for Finn
    """

    # ── Spending context ──────────────────────────────────────
    if analysis is not None:
        spending_context = f"""
STUDENT'S CURRENT FINANCIAL DATA:
- Income:               ${analysis['total_income']:.2f}
- Essential expenses:   ${analysis['total_essential']:.2f} ({analysis['essential_pct']:.1f}% of income)
- Discretionary spend:  ${analysis['net_discretionary']:.2f}
- Net position:         ${analysis['net_position']:.2f}
- Health score:         {analysis['health_score']}/100 ({analysis['health_label']})
- Invisible spending:   ${analysis['invisible_total']:.2f} ({analysis['invisible_count']} transactions)
- Subscriptions:        ${analysis['subscription_total']:.2f}/month
- Social spending:      ${analysis['social_total']:.2f} ({analysis['social_pct']:.1f}%) — {analysis['social_label']}

Top spending categories:
"""
        for cat, row in analysis["category_summary"].head(5).iterrows():
            cat_type = "Essential" if row["essential"] else "Flexible"
            spending_context += (
                f"  {cat}: ${row['total']:.2f} "
                f"({row['percentage']:.1f}%) [{cat_type}]\n"
            )
    else:
        spending_context = (
            "No spending data loaded. If the student asks about their specific "
            "finances, tell them to upload their CSV first."
        )

    # ── Budget context ────────────────────────────────────────
    if budget_plan is not None:
        budget_context = f"""
STUDENT'S BUDGET PLAN:
- Monthly discretionary budget: ${budget_plan.get('monthly_budget', 0):.2f}
- Safety net target:            ${budget_plan.get('safety_net_target', 0):.2f}
- Safety net current:           ${budget_plan.get('safety_net_current', 0):.2f}
- Savings goals: {budget_plan.get('goals', [])}

Category limits set by student:
"""
        for cat, limit in budget_plan.get("category_limits", {}).items():
            budget_context += f"  {cat}: ${limit:.2f}/month\n"
    else:
        budget_context = "No budget plan set yet."

    return f"""
You are Finn, a financial advisor for Australian university students.

YOUR PERSONALITY:
- Direct and honest. You do not soften bad financial news.
- You give specific, actionable advice referenced to real numbers.
- You are not unkind, but you do not use empty encouragement.
- You understand that uni students have a social life and small treats like
  coffee are acceptable — focus advice on high-impact changes, not petty ones.
- You use Australian context: Centrelink, Transperth, Woolies, Perth venues.
- You keep responses concise — 4 to 6 sentences unless a breakdown is requested.

YOUR RULES:
- Never suggest reducing essential expenses: rent, utilities, phone bills.
- Always acknowledge that some social spending is healthy — do not tell students
  to stop going out entirely.
- Prioritise food delivery and eating out as savings targets before social spending.
- Social spending is only a concern if it exceeds 35% of discretionary income.
- Small purchases like 1-2 coffees per week are not worth flagging — focus on
  habitual convenience store visits and vending machines instead.
- When a student is in deficit, say so clearly and focus on what they can change.
- Reference the student's real numbers in every answer where possible.
- If asked something outside personal finance, redirect back to their finances.
- If the student pushes back on your advice, hold your position if the data supports it.

{spending_context}
{budget_context}
"""


def chat_with_finn(message: str, history: list,
                   analysis: dict = None,
                   budget_plan: dict = None) -> str:
    """
    Processes a chat message and returns Finn's response.

    Args:
        message:     student's message
        history:     Gradio chat history list
        analysis:    current spending analysis
        budget_plan: current budget plan
    Returns:
        str: Finn's response
    """

    system_prompt = build_finn_system_prompt(analysis, budget_plan)

    # build conversation history string
    history_text = ""
    for exchange in history[-6:]:
        if isinstance(exchange, dict):
            role = "Student" if exchange["role"] == "user" else "Finn"
            history_text += f"{role}: {exchange['content']}\n"
        elif isinstance(exchange, (list, tuple)) and len(exchange) == 2:
            history_text += f"Student: {exchange[0]}\nFinn: {exchange[1]}\n"

    full_prompt = (
        f"{system_prompt}\n\n"
        f"Conversation so far:\n{history_text}\n"
        f"Student: {message}\n"
        f"Finn:"
    )

    try:
        return get_response(full_prompt)
    except Exception as e:
        return f"Could not reach AI: {e}"


print("Cell 8 ready — AI recommendations and Finn chatbot defined.")

Cell 8 ready — AI recommendations and Finn chatbot defined.


## Gradio UI Integration

In [45]:
# ── Cell 9: Gradio UI — Smart Finance Assistant ───────────────

import plotly.graph_objects as go
import plotly.express as px

# ── Colour palette ────────────────────────────────────────────
COLOURS = {
    "navy":        "#1B2A4A",
    "charcoal":    "#2D3748",
    "green":       "#2D6A4F",
    "green_light": "#52B788",
    "red":         "#C1121F",
    "red_light":   "#E63946",
    "amber":       "#E07A00",
    "white":       "#FFFFFF",
    "off_white":   "#F8F9FA",
    "border":      "#E2E8F0",
    "text_muted":  "#718096",
}

CATEGORY_COLOURS = [
    "#2D6A4F","#52B788","#1B4332","#40916C",
    "#74C69D","#95D5B2","#B7E4C7","#D8F3DC",
    "#1B2A4A","#2D3748"
]

# ── Chart builders ────────────────────────────────────────────

def build_donut_chart(category_summary):
    labels  = category_summary.index.tolist()
    values  = category_summary["total"].tolist()
    colours = CATEGORY_COLOURS[:len(labels)]
    fig = go.Figure(go.Pie(
        labels        = labels,
        values        = values,
        hole          = 0.55,
        marker_colors = colours,
        textinfo      = "label+percent",
        hovertemplate = "<b>%{label}</b><br>$%{value:.2f}<br>%{percent}<extra></extra>",
        textfont_size = 12,
    ))
    fig.update_layout(
        paper_bgcolor = "rgba(0,0,0,0)",
        plot_bgcolor  = "rgba(0,0,0,0)",
        showlegend    = False,
        margin        = dict(t=30, b=10, l=10, r=10),
        height        = 340,
        annotations   = [dict(
            text       = "Spending<br>Breakdown",
            x=0.5, y=0.5,
            font_size  = 13,
            font_color = COLOURS["charcoal"],
            showarrow  = False
        )]
    )
    return fig


def build_bar_chart(category_summary):
    df_plot = category_summary.sort_values("total", ascending=True)
    colours = [COLOURS["red"] if not row["essential"]
               else COLOURS["navy"]
               for _, row in df_plot.iterrows()]
    fig = go.Figure(go.Bar(
        x             = df_plot["total"],
        y             = df_plot.index,
        orientation   = "h",
        marker_color  = colours,
        hovertemplate = "<b>%{y}</b><br>$%{x:.2f}<extra></extra>",
        text          = [f"${v:.0f}" for v in df_plot["total"]],
        textposition  = "outside",
    ))
    fig.update_layout(
        paper_bgcolor = "rgba(0,0,0,0)",
        plot_bgcolor  = "rgba(0,0,0,0)",
        xaxis_title   = "Amount ($)",
        xaxis         = dict(showgrid=True, gridcolor=COLOURS["border"]),
        yaxis         = dict(showgrid=False),
        margin        = dict(t=20, b=40, l=10, r=60),
        height        = 380,
        font          = dict(color=COLOURS["charcoal"]),
    )
    return fig


def build_day_chart(day_spend):
    days    = day_spend.index.tolist()
    amounts = day_spend.values.tolist()
    max_val = max(amounts) if amounts else 1
    colours = [COLOURS["red"] if v == max_val
               else COLOURS["green_light"] for v in amounts]
    fig = go.Figure(go.Bar(
        x             = days,
        y             = amounts,
        marker_color  = colours,
        hovertemplate = "<b>%{x}</b><br>$%{y:.2f}<extra></extra>",
        text          = [f"${v:.0f}" for v in amounts],
        textposition  = "outside",
    ))
    fig.update_layout(
        paper_bgcolor = "rgba(0,0,0,0)",
        plot_bgcolor  = "rgba(0,0,0,0)",
        xaxis_title   = "Day of Week",
        yaxis_title   = "Amount ($)",
        yaxis         = dict(showgrid=True, gridcolor=COLOURS["border"]),
        margin        = dict(t=20, b=40, l=10, r=10),
        height        = 300,
        font          = dict(color=COLOURS["charcoal"]),
    )
    return fig


def build_health_gauge(score, label):
    if score >= 80:
        colour = COLOURS["green"]
    elif score >= 60:
        colour = COLOURS["amber"]
    elif score >= 40:
        colour = COLOURS["red_light"]
    else:
        colour = COLOURS["red"]
    fig = go.Figure(go.Indicator(
        mode  = "gauge+number",
        value = score,
        title = dict(text=f"Financial Health — {label}",
                     font=dict(size=14, color=COLOURS["charcoal"])),
        gauge = dict(
            axis        = dict(range=[0, 100], tickwidth=1,
                               tickcolor=COLOURS["charcoal"]),
            bar         = dict(color=colour),
            bgcolor     = COLOURS["off_white"],
            bordercolor = COLOURS["border"],
            steps       = [
                dict(range=[0,  40], color="#FEE2E2"),
                dict(range=[40, 60], color="#FEF3C7"),
                dict(range=[60, 80], color="#D1FAE5"),
                dict(range=[80,100], color="#A7F3D0"),
            ],
            threshold = dict(
                line  = dict(color=COLOURS["charcoal"], width=3),
                value = score
            )
        ),
        number = dict(suffix="/100", font=dict(size=28))
    ))
    fig.update_layout(
        paper_bgcolor = "rgba(0,0,0,0)",
        margin        = dict(t=40, b=10, l=20, r=20),
        height        = 260,
    )
    return fig


def build_trend_chart(snapshots):
    if len(snapshots) < 2:
        return None
    labels = [s.get("label", f"Month {i+1}")
              for i, s in enumerate(snapshots)]
    scores = [s.get("health_score", 0) for s in snapshots]
    fig = go.Figure(go.Scatter(
        x    = labels,
        y    = scores,
        mode = "lines+markers",
        line = dict(color=COLOURS["green"], width=3),
        marker = dict(size=9, color=COLOURS["navy"]),
        hovertemplate = "<b>%{x}</b><br>Score: %{y}/100<extra></extra>",
    ))
    fig.update_layout(
        paper_bgcolor = "rgba(0,0,0,0)",
        plot_bgcolor  = "rgba(0,0,0,0)",
        xaxis_title   = "Month",
        yaxis_title   = "Health Score",
        yaxis         = dict(range=[0, 100], showgrid=True,
                             gridcolor=COLOURS["border"]),
        margin        = dict(t=20, b=40, l=10, r=10),
        height        = 280,
        font          = dict(color=COLOURS["charcoal"]),
    )
    return fig


def build_savings_progress_chart(goals):
    if not goals:
        return None
    names    = [g["name"]           for g in goals]
    targets  = [g["target"]         for g in goals]
    currents = [g.get("current", 0) for g in goals]
    pcts     = [min(100, c / t * 100) if t > 0 else 0
                for c, t in zip(currents, targets)]
    fig = go.Figure()
    for name, pct, current, target in zip(names, pcts, currents, targets):
        fig.add_trace(go.Bar(
            x             = [pct],
            y             = [name],
            orientation   = "h",
            marker_color  = COLOURS["green_light"],
            hovertemplate = (f"<b>{name}</b><br>${current:.0f} of "
                             f"${target:.0f}<br>{pct:.1f}%<extra></extra>"),
            text          = [f"{pct:.0f}%  ${current:.0f} / ${target:.0f}"],
            textposition  = "inside",
            name          = name,
        ))
    fig.update_layout(
        paper_bgcolor = "rgba(0,0,0,0)",
        plot_bgcolor  = "rgba(0,0,0,0)",
        xaxis         = dict(range=[0, 100], title="Progress (%)"),
        yaxis         = dict(showgrid=False),
        barmode       = "overlay",
        showlegend    = False,
        margin        = dict(t=20, b=40, l=10, r=10),
        height        = max(200, len(goals) * 80),
        font          = dict(color=COLOURS["charcoal"]),
    )
    return fig


def build_budget_vs_actual_chart(plan, analysis):
    if plan is None or analysis is None:
        return None
    limits  = plan.get("category_limits", {})
    cats    = list(limits.keys())
    budgets = [limits[c] for c in cats]
    actuals = []
    for c in cats:
        if c in analysis["category_summary"].index:
            actuals.append(analysis["category_summary"].loc[c, "total"])
        else:
            actuals.append(0)
    colours = [COLOURS["red"] if a > b else COLOURS["green"]
               for a, b in zip(actuals, budgets)]
    fig = go.Figure()
    fig.add_trace(go.Bar(
        name          = "Budget Limit",
        x             = cats,
        y             = budgets,
        marker_color  = COLOURS["navy"],
        opacity       = 0.4,
        hovertemplate = "<b>%{x}</b><br>Budget: $%{y:.2f}<extra></extra>",
    ))
    fig.add_trace(go.Bar(
        name          = "Actual Spend",
        x             = cats,
        y             = actuals,
        marker_color  = colours,
        hovertemplate = "<b>%{x}</b><br>Actual: $%{y:.2f}<extra></extra>",
    ))
    fig.update_layout(
        paper_bgcolor = "rgba(0,0,0,0)",
        plot_bgcolor  = "rgba(0,0,0,0)",
        barmode       = "overlay",
        xaxis_title   = "Category",
        yaxis_title   = "Amount ($)",
        yaxis         = dict(showgrid=True, gridcolor=COLOURS["border"]),
        legend        = dict(orientation="h", y=1.1),
        margin        = dict(t=40, b=60, l=10, r=10),
        height        = 340,
        font          = dict(color=COLOURS["charcoal"]),
    )
    return fig


# ── CSS ───────────────────────────────────────────────────────
CSS = """
.gradio-container {
    font-family: 'Inter', -apple-system, BlinkMacSystemFont, sans-serif !important;
    background-color: #F8F9FA !important;
}
.card {
    background: white;
    border-radius: 10px;
    padding: 20px;
    border: 1px solid #E2E8F0;
    margin-bottom: 12px;
}
.metric-value {
    font-size: 28px;
    font-weight: 700;
    color: #1B2A4A;
}
.metric-label {
    font-size: 13px;
    color: #718096;
    text-transform: uppercase;
    letter-spacing: 0.05em;
}
.insight-critical {
    background: #FEE2E2;
    border-left: 4px solid #C1121F;
    padding: 12px 16px;
    border-radius: 6px;
    margin: 8px 0;
}
.insight-warning {
    background: #FEF3C7;
    border-left: 4px solid #E07A00;
    padding: 12px 16px;
    border-radius: 6px;
    margin: 8px 0;
}
.insight-info {
    background: #EBF8FF;
    border-left: 4px solid #2B6CB0;
    padding: 12px 16px;
    border-radius: 6px;
    margin: 8px 0;
}
h1, h2, h3 { color: #1B2A4A !important; }
.step-header {
    font-size: 22px;
    font-weight: 700;
    color: #1B2A4A;
    border-bottom: 2px solid #E2E8F0;
    padding-bottom: 10px;
    margin-bottom: 16px;
}
"""


# ── Helpers ───────────────────────────────────────────────────
def insights_to_html(insights: list) -> str:
    if not insights:
        return "<p>No insights generated.</p>"
    html = ""
    for ins in insights:
        css_class = f"insight-{ins.get('severity', 'info')}"
        html += (
            f'<div class="{css_class}">'
            f'<strong>{ins["title"]}</strong><br>'
            f'{ins["detail"]}'
            f'</div>'
        )
    return html


def overview_to_html(analysis: dict) -> str:
    pos        = analysis["net_position"]
    pos_colour = COLOURS["red"] if pos < 0 else COLOURS["green"]
    pos_label  = "DEFICIT" if pos < 0 else "Surplus"
    return f"""
<div style="display:grid; grid-template-columns:repeat(4,1fr);
            gap:16px; margin-bottom:16px;">
  <div class="card" style="text-align:center;">
    <div class="metric-label">Total Income</div>
    <div class="metric-value" style="color:{COLOURS['navy']};">
      ${analysis['total_income']:.2f}</div>
  </div>
  <div class="card" style="text-align:center;">
    <div class="metric-label">Essential Expenses</div>
    <div class="metric-value" style="color:{COLOURS['charcoal']};">
      ${analysis['total_essential']:.2f}</div>
    <div style="font-size:12px; color:{COLOURS['text_muted']};">
      {analysis['essential_pct']:.1f}% of income</div>
  </div>
  <div class="card" style="text-align:center;">
    <div class="metric-label">Discretionary Spend</div>
    <div class="metric-value" style="color:{COLOURS['charcoal']};">
      ${analysis['net_discretionary']:.2f}</div>
  </div>
  <div class="card" style="text-align:center;">
    <div class="metric-label">Net Position</div>
    <div class="metric-value" style="color:{pos_colour};">
      ${pos:.2f}</div>
    <div style="font-size:12px; color:{pos_colour}; font-weight:600;">
      {pos_label}</div>
  </div>
</div>
"""


def budget_comparison_to_html(plan: dict, analysis: dict) -> str:
    if plan is None or analysis is None:
        return ""
    limits = plan.get("category_limits", {})
    rows   = ""
    for cat, limit in limits.items():
        actual = 0
        if cat in analysis["category_summary"].index:
            actual = analysis["category_summary"].loc[cat, "total"]
        diff        = limit - actual
        diff_colour = COLOURS["green"] if diff >= 0 else COLOURS["red"]
        diff_label  = f"+${diff:.2f}" if diff >= 0 else f"-${abs(diff):.2f}"
        status      = "Within limit" if diff >= 0 else "Over limit"
        rows += f"""
<tr style="border-bottom:1px solid {COLOURS['border']};">
  <td style="padding:8px;">{cat}</td>
  <td style="padding:8px; text-align:right;">${limit:.2f}</td>
  <td style="padding:8px; text-align:right;">${actual:.2f}</td>
  <td style="padding:8px; text-align:right; color:{diff_colour};
             font-weight:600;">{diff_label}</td>
  <td style="padding:8px; text-align:center; color:{diff_colour};
             font-size:12px;">{status}</td>
</tr>"""
    return f"""
<div class="card">
  <div style="font-weight:600; font-size:15px; color:{COLOURS['navy']};
              margin-bottom:12px;">This Month vs Your Budget</div>
  <table style="width:100%; border-collapse:collapse; font-size:14px;">
    <thead>
      <tr style="background:{COLOURS['off_white']};
                 border-bottom:2px solid {COLOURS['border']};">
        <th style="padding:8px; text-align:left;">Category</th>
        <th style="padding:8px; text-align:right;">Budget Limit</th>
        <th style="padding:8px; text-align:right;">Actual Spend</th>
        <th style="padding:8px; text-align:right;">Difference</th>
        <th style="padding:8px; text-align:center;">Status</th>
      </tr>
    </thead>
    <tbody>{rows}</tbody>
  </table>
</div>
"""


def nav_html(current_step: int, max_reached: int) -> str:
    steps = [
        "1. Import", "2. Edit", "3. Insights",
        "4. Plan", "5. Budget", "6. Chat"
    ]
    buttons = ""
    for i, label in enumerate(steps, start=1):
        if i == current_step:
            style = (f"padding:8px 16px; border-radius:20px; font-size:13px; "
                     f"font-weight:600; border:2px solid {COLOURS['navy']}; "
                     f"background:{COLOURS['navy']}; color:white; cursor:pointer;")
        elif i <= max_reached:
            style = (f"padding:8px 16px; border-radius:20px; font-size:13px; "
                     f"font-weight:600; border:2px solid {COLOURS['green']}; "
                     f"background:white; color:{COLOURS['green']}; cursor:pointer;")
        else:
            style = (f"padding:8px 16px; border-radius:20px; font-size:13px; "
                     f"font-weight:600; border:2px solid {COLOURS['border']}; "
                     f"background:white; color:{COLOURS['text_muted']}; "
                     f"cursor:not-allowed;")
        buttons += f'<span style="{style}">{label}</span>'
    return f'<div style="display:flex; gap:8px; margin-bottom:20px; flex-wrap:wrap;">{buttons}</div>'


# ── Main UI builder ───────────────────────────────────────────
def create_finance_assistant_ui():

    nav_state = {"max_reached": 1}

    # ── Handlers ──────────────────────────────────────────────

    def handle_upload(file):
        if file is None:
            return (gr.update(visible=True), gr.update(visible=False),
                    "No file uploaded.", None, nav_html(1, 1))
        try:
            app_state["current_df"]       = None
            app_state["current_analysis"] = None
            df = load_and_clean_transaction_data(file.name)
            df = categorise_transactions(df)
            df = flag_essential_transactions(df)
            app_state["current_df"]    = df
            app_state["current_label"] = get_snapshot_label(df)
            preview = df[["date","description","amount",
                          "category","essential"]].head(10).copy()
            preview["date"] = preview["date"].dt.strftime("%d/%m/%Y")
            nav_state["max_reached"] = max(nav_state["max_reached"], 2)
            return (gr.update(visible=False), gr.update(visible=True),
                    f"Loaded {len(df)} transactions — {app_state['current_label']}",
                    preview, nav_html(2, nav_state["max_reached"]))
        except Exception as e:
            return (gr.update(visible=True), gr.update(visible=False),
                    f"Error loading file: {e}", None, nav_html(1, 1))

    def handle_goto_step2():
        nav_state["max_reached"] = max(nav_state["max_reached"], 2)
        df = app_state["current_df"]
        if df is not None:
            display_df = df[["date","description","amount",
                              "category","essential"]].copy()
            display_df["date"] = display_df["date"].dt.strftime("%d/%m/%Y")
        else:
            display_df = None
        return (
            gr.update(visible=False),
            gr.update(visible=True),
            display_df,
            nav_html(2, nav_state["max_reached"])
        )

    def handle_add_transaction(date_str, description, amount, category, essential):
        if app_state["current_df"] is None:
            return "Please upload a CSV file first.", None
        try:
            new_row = pd.DataFrame([{
                "date":        pd.to_datetime(date_str, dayfirst=True),
                "description": description.upper(),
                "amount":      float(amount),
                "category":    category,
                "essential":   essential
            }])
            app_state["current_df"] = pd.concat(
                [app_state["current_df"], new_row], ignore_index=True
            ).sort_values("date").reset_index(drop=True)
            display_df = app_state["current_df"][
                ["date","description","amount","category","essential"]
            ].copy()
            display_df["date"] = display_df["date"].dt.strftime("%d/%m/%Y")
            return "Transaction added.", display_df
        except Exception as e:
            return f"Error: {e}", None

    def handle_save_edits(df):
        try:
            app_state["current_df"] = pd.DataFrame(df)
            app_state["current_df"]["date"] = pd.to_datetime(
                app_state["current_df"]["date"], dayfirst=True)
            app_state["current_df"]["amount"] = pd.to_numeric(
                app_state["current_df"]["amount"], errors="coerce")
            return "Changes saved."
        except Exception as e:
            return f"Error saving: {e}"

    def handle_run_analysis():
        if app_state["current_df"] is None:
            return (gr.update(visible=False), gr.update(visible=True),
                    None, None, None, None,
                    "<p>No data loaded.</p>", "<p></p>",
                    gr.update(visible=False), None,
                    gr.update(visible=False), None, None,
                    nav_html(2, nav_state["max_reached"]))
        try:
            analysis = analyse_spending_by_category(app_state["current_df"])
            app_state["current_analysis"] = analysis
            nav_state["max_reached"] = max(nav_state["max_reached"], 3)

            donut      = build_donut_chart(analysis["category_summary"])
            bar        = build_bar_chart(analysis["category_summary"])
            day        = build_day_chart(analysis["day_spend"])
            gauge      = build_health_gauge(analysis["health_score"],
                                            analysis["health_label"])
            overview_h = overview_to_html(analysis)
            insights_h = insights_to_html(analysis["insights"])

            trend = None
            if len(app_state["monthly_snapshots"]) >= 1:
                all_snaps = app_state["monthly_snapshots"] + [{
                    "label":        app_state["current_label"],
                    "health_score": analysis["health_score"]
                }]
                trend = build_trend_chart(all_snaps)

            bva_html_val  = ""
            bva_chart_val = None
            returning     = app_state.get("budget_plan") is not None
            if returning:
                bva_html_val  = budget_comparison_to_html(
                    app_state["budget_plan"], analysis)
                bva_chart_val = build_budget_vs_actual_chart(
                    app_state["budget_plan"], analysis)

            return (
                gr.update(visible=True),
                gr.update(visible=False),
                donut, bar, day, gauge,
                overview_h, insights_h,
                gr.update(visible=True) if trend else gr.update(visible=False),
                trend,
                gr.update(visible=returning),
                bva_html_val,
                bva_chart_val,
                nav_html(3, nav_state["max_reached"])
            )
        except Exception as e:
            return (gr.update(visible=False), gr.update(visible=True),
                    None, None, None, None,
                    f"<p>Analysis error: {e}</p>", "<p></p>",
                    gr.update(visible=False), None,
                    gr.update(visible=False), None, None,
                    nav_html(2, nav_state["max_reached"]))

    def handle_generate_advice():
        if app_state["current_analysis"] is None:
            return "Run analysis first."
        return generate_ai_recommendations(app_state["current_analysis"])

    def handle_save_budget(monthly_budget, safety_target, safety_current,
                           goal1_name, goal1_target, goal1_current,
                           goal2_name, goal2_target, goal2_current,
                           food_limit, coffee_limit, eating_limit,
                           transport_limit, social_limit, shopping_limit,
                           subs_limit):
        goals = []
        if goal1_name and goal1_target > 0:
            goals.append({"name": goal1_name, "target": goal1_target,
                          "current": goal1_current})
        if goal2_name and goal2_target > 0:
            goals.append({"name": goal2_name, "target": goal2_target,
                          "current": goal2_current})

        budget_plan = {
            "monthly_budget":     monthly_budget,
            "safety_net_target":  safety_target,
            "safety_net_current": safety_current,
            "goals":              goals,
            "category_limits": {
                "Food & Groceries": food_limit,
                "Coffee & Snacks":  coffee_limit,
                "Eating Out":       eating_limit,
                "Transport":        transport_limit,
                "Social":           social_limit,
                "Shopping":         shopping_limit,
                "Subscriptions":    subs_limit,
            }
        }
        app_state["budget_plan"]   = budget_plan
        app_state["safety_net"]    = {"target": safety_target,
                                      "current": safety_current}
        app_state["savings_goals"] = goals

        if app_state["current_analysis"] is not None:
            snapshot = {
                "label":        app_state["current_label"],
                "health_score": app_state["current_analysis"]["health_score"],
                "net_position": app_state["current_analysis"]["net_position"],
                "total_income": app_state["current_analysis"]["total_income"],
                "budget_plan":  budget_plan,
            }
            existing = [s["label"] for s in app_state["monthly_snapshots"]]
            if app_state["current_label"] not in existing:
                app_state["monthly_snapshots"].append(snapshot)

        save_app_data(app_state)
        nav_state["max_reached"] = max(nav_state["max_reached"], 5)

        summary    = build_budget_summary_html(
            budget_plan, app_state["current_analysis"])
        prog_chart = build_savings_progress_chart(goals) if goals else None

        return (
            "Budget saved.",
            summary,
            prog_chart if prog_chart else go.Figure(),
            gr.update(visible=True if prog_chart else False),
            nav_html(5, nav_state["max_reached"])
        )

    def build_budget_summary_html(plan, analysis):
        if plan is None:
            return "<p>No budget set.</p>"
        rows = ""
        for cat, limit in plan["category_limits"].items():
            actual = 0
            if (analysis is not None and
                    cat in analysis["category_summary"].index):
                actual = analysis["category_summary"].loc[cat, "total"]
            diff        = limit - actual
            diff_colour = COLOURS["green"] if diff >= 0 else COLOURS["red"]
            diff_label  = f"+${diff:.2f}" if diff >= 0 else f"-${abs(diff):.2f}"
            rows += f"""
<tr style="border-bottom:1px solid {COLOURS['border']};">
  <td style="padding:8px;">{cat}</td>
  <td style="padding:8px; text-align:right;">${limit:.2f}</td>
  <td style="padding:8px; text-align:right;">${actual:.2f}</td>
  <td style="padding:8px; text-align:right; color:{diff_colour};
             font-weight:600;">{diff_label}</td>
</tr>"""

        safety_pct = (plan["safety_net_current"] /
                      plan["safety_net_target"] * 100
                      if plan["safety_net_target"] > 0 else 0)

        return f"""
<div class="card">
  <div class="step-header">Your Budget Plan</div>
  <p><strong>Monthly Discretionary Budget:</strong>
     ${plan['monthly_budget']:.2f}</p>
  <h3 style="margin-top:16px;">Safety Net</h3>
  <p>Target: ${plan['safety_net_target']:.2f} &nbsp;|&nbsp;
     Saved: ${plan['safety_net_current']:.2f} &nbsp;|&nbsp;
     Progress: {safety_pct:.1f}%</p>
  <div style="background:{COLOURS['border']}; border-radius:4px; height:12px;">
    <div style="background:{COLOURS['green']};
                width:{min(100, safety_pct):.1f}%;
                height:12px; border-radius:4px;"></div>
  </div>
  <h3 style="margin-top:20px;">Category Limits vs Past Spending</h3>
  <table style="width:100%; border-collapse:collapse; font-size:14px;">
    <thead>
      <tr style="background:{COLOURS['off_white']};
                 border-bottom:2px solid {COLOURS['border']};">
        <th style="padding:8px; text-align:left;">Category</th>
        <th style="padding:8px; text-align:right;">Your Limit</th>
        <th style="padding:8px; text-align:right;">Past Spend</th>
        <th style="padding:8px; text-align:right;">Difference</th>
      </tr>
    </thead>
    <tbody>{rows}</tbody>
  </table>
  <p style="font-size:12px; color:{COLOURS['text_muted']}; margin-top:8px;">
    Positive difference means your limit is above past spending.
    Negative means you need to reduce spending to meet your limit.
  </p>
</div>
"""

    def handle_clear_all(confirm):
        if confirm != "YES":
            return ("Type YES in the confirmation box to clear all data.",
                    nav_html(1, 1))
        app_state["budget_plan"]       = None
        app_state["safety_net"]        = None
        app_state["savings_goals"]     = []
        app_state["monthly_snapshots"] = []
        app_state["current_df"]        = None
        app_state["current_analysis"]  = None
        app_state["current_label"]     = None
        nav_state["max_reached"]       = 1
        if Path(SAVE_FILE).exists():
            Path(SAVE_FILE).unlink()
        return ("All data cleared. Refresh the page to start fresh.",
                nav_html(1, 1))

    def handle_finn_chat(message, history):
        return chat_with_finn(
            message, history,
            analysis    = app_state.get("current_analysis"),
            budget_plan = app_state.get("budget_plan")
        )

    # ── Pre-populate from saved state ─────────────────────────
    saved_plan   = app_state.get("budget_plan") or {}
    saved_limits = saved_plan.get("category_limits", {})
    saved_goals  = saved_plan.get("goals", [])
    goal1        = saved_goals[0] if len(saved_goals) > 0 else {}
    goal2        = saved_goals[1] if len(saved_goals) > 1 else {}

    if saved_plan:
        nav_state["max_reached"] = 5

    # ── Build UI ──────────────────────────────────────────────
    with gr.Blocks(css=CSS, title="Smart Finance Assistant") as app:

        gr.HTML(f"""
<div style="background:{COLOURS['navy']}; padding:24px 32px;
            border-radius:10px; margin-bottom:16px;">
  <h1 style="color:white; margin:0; font-size:26px; font-weight:700;">
    Smart Finance Assistant
  </h1>
  <p style="color:#A0AEC0; margin:4px 0 0; font-size:14px;">
    Personal finance analysis and budgeting for university students
  </p>
</div>
""")

        nav_bar = gr.HTML(nav_html(1, nav_state["max_reached"]))

        # ════════════════════════════════════════════════════
        # STEP 1 — IMPORT DATA
        # ════════════════════════════════════════════════════
        with gr.Group() as step1_group:
            gr.HTML('<div class="step-header">Step 1 — Import Your Data</div>')
            gr.HTML("""
<div class="card">
  <p style="margin:0; color:#4A5568;">
    Upload a CSV export from your bank. The file needs at least three columns:
    <strong>Date</strong>, <strong>Description</strong>, and
    <strong>Amount</strong>. Categories are assigned automatically.
  </p>
</div>""")
            csv_file      = gr.File(label="Bank CSV File", file_types=[".csv"])
            upload_status = gr.Textbox(label="Status", interactive=False)
            preview_table = gr.Dataframe(label="Preview (first 10 rows)",
                                         interactive=False)
            next_to_step2 = gr.Button("Continue to Review & Edit →",
                                      variant="primary")

        # ════════════════════════════════════════════════════
        # STEP 2 — REVIEW & EDIT
        # ════════════════════════════════════════════════════
        with gr.Group(visible=False) as step2_group:
            gr.HTML('<div class="step-header">Step 2 — Review and Edit Transactions</div>')
            gr.HTML("""
<div class="card">
  <p style="margin:0; color:#4A5568;">
    Review the transactions below. Correct any categories and mark essential
    items such as rent, power, and phone bills. Essential expenses are tracked
    separately and will never be targeted for savings advice.
  </p>
</div>""")
            edit_table  = gr.Dataframe(label="All Transactions",
                                       interactive=True, wrap=True)
            save_status = gr.Textbox(label="Save Status", interactive=False)
            save_btn    = gr.Button("Save Edits", variant="secondary")

            gr.HTML('<div style="margin-top:20px; font-weight:600; '
                    'font-size:15px; color:#1B2A4A;">Add a Cash Transaction</div>')
            with gr.Row():
                m_date = gr.Textbox(label="Date (DD/MM/YYYY)",
                                    placeholder="15/03/2025")
                m_desc = gr.Textbox(label="Description",
                                    placeholder="Cash — markets")
                m_amt  = gr.Number(label="Amount (negative = expense)",
                                   value=-20.00)
            with gr.Row():
                m_cat = gr.Dropdown(label="Category",
                                    choices=CATEGORIES, value="Other")
                m_ess = gr.Checkbox(label="Essential", value=False)
            add_btn    = gr.Button("Add Transaction", variant="secondary")
            add_status = gr.Textbox(label="Status", interactive=False)

            with gr.Row():
                back_to_step1 = gr.Button("← Back to Import",
                                          variant="secondary")
                next_to_step3 = gr.Button("Run Analysis →",
                                          variant="primary")

        # ════════════════════════════════════════════════════
        # STEP 3 — SPENDING INSIGHTS
        # ════════════════════════════════════════════════════
        with gr.Group(visible=False) as step3_group:
            gr.HTML('<div class="step-header">Step 3 — Spending Insights</div>')

            overview_html = gr.HTML()
            insights_html = gr.HTML()

            with gr.Row():
                donut_chart = gr.Plot(label="Spending Breakdown")
                gauge_chart = gr.Plot(label="Financial Health Score")

            with gr.Row():
                bar_chart = gr.Plot(label="Spending by Category")
                day_chart = gr.Plot(label="Spending by Day of Week")

            with gr.Group(visible=False) as trend_group:
                gr.HTML('<div style="font-weight:600; font-size:15px; '
                        'color:#1B2A4A; margin:16px 0 8px;">'
                        'Health Score Trend</div>')
                trend_chart = gr.Plot(label="Month on Month Trend")

            with gr.Group(visible=False) as bva_group:
                gr.HTML('<div style="font-weight:600; font-size:15px; '
                        'color:#1B2A4A; margin:16px 0 8px;">'
                        'This Month vs Your Budget</div>')
                bva_html_out = gr.HTML()
                bva_chart    = gr.Plot(label="Budget vs Actual")

            advice_btn = gr.Button("Generate AI Recommendations",
                                   variant="primary")
            advice_out = gr.Textbox(label="AI Recommendations",
                                    interactive=False, lines=18)

            with gr.Row():
                back_to_step2   = gr.Button("← Back to Edit",
                                            variant="secondary")
                next_from_step3 = gr.Button(
                    "View Budget Performance →"
                    if saved_plan else "Plan My Budget →",
                    variant="primary"
                )

        # ════════════════════════════════════════════════════
        # STEP 4 — PLAN BUDGET
        # ════════════════════════════════════════════════════
        with gr.Group(visible=False) as step4_group:
            gr.HTML('<div class="step-header">Step 4 — Plan Your Budget</div>')
            gr.HTML("""
<div class="card">
  <p style="margin:0; color:#4A5568;">
    Set your forward-looking budget based on what you have just seen.
    Set limits based on what you can realistically achieve,
    not what looks good on paper.
  </p>
</div>""")
            gr.HTML('<div style="font-weight:600; font-size:15px; '
                    'color:#1B2A4A; margin:16px 0 8px;">Monthly Overview</div>')
            monthly_budget = gr.Slider(
                minimum=200, maximum=3000, step=50, value=800,
                label="Total Monthly Discretionary Budget ($)"
            )
            gr.HTML('<div style="font-weight:600; font-size:15px; '
                    'color:#1B2A4A; margin:16px 0 8px;">Safety Net</div>')
            gr.HTML("""
<div class="card insight-warning">
  <strong>Build your safety net before saving for goals.</strong>
  As a casual worker you have no sick leave. One missed week can mean not
  making rent. Set a target and track your progress here.
</div>""")
            with gr.Row():
                safety_target  = gr.Number(
                    label="Safety Net Target ($)",
                    value=saved_plan.get("safety_net_target", 1000))
                safety_current = gr.Number(
                    label="Amount Currently Saved ($)",
                    value=saved_plan.get("safety_net_current", 0))

            gr.HTML('<div style="font-weight:600; font-size:15px; '
                    'color:#1B2A4A; margin:16px 0 8px;">Savings Goals</div>')
            with gr.Row():
                goal1_name    = gr.Textbox(label="Goal 1 Name",
                                           placeholder="Bali trip",
                                           value=goal1.get("name", ""))
                goal1_target  = gr.Number(label="Target ($)",
                                          value=goal1.get("target", 0))
                goal1_current = gr.Number(label="Already Saved ($)",
                                          value=goal1.get("current", 0))
            with gr.Row():
                goal2_name    = gr.Textbox(label="Goal 2 Name",
                                           placeholder="New laptop",
                                           value=goal2.get("name", ""))
                goal2_target  = gr.Number(label="Target ($)",
                                          value=goal2.get("target", 0))
                goal2_current = gr.Number(label="Already Saved ($)",
                                          value=goal2.get("current", 0))

            gr.HTML('<div style="font-weight:600; font-size:15px; '
                    'color:#1B2A4A; margin:16px 0 8px;">'
                    'Monthly Spending Limits by Category</div>')
            gr.HTML("""
<div class="card">
  <p style="margin:0; color:#4A5568; font-size:13px;">
    Set a realistic monthly limit for each discretionary category.
    These will be compared against your actual spending each month.
  </p>
</div>""")
            with gr.Row():
                food_limit      = gr.Number(
                    label="Food & Groceries ($)",
                    value=saved_limits.get("Food & Groceries", 250))
                coffee_limit    = gr.Number(
                    label="Coffee & Snacks ($)",
                    value=saved_limits.get("Coffee & Snacks", 30))
                eating_limit    = gr.Number(
                    label="Eating Out ($)",
                    value=saved_limits.get("Eating Out", 80))
                transport_limit = gr.Number(
                    label="Transport ($)",
                    value=saved_limits.get("Transport", 80))
            with gr.Row():
                social_limit   = gr.Number(
                    label="Social ($)",
                    value=saved_limits.get("Social", 120))
                shopping_limit = gr.Number(
                    label="Shopping ($)",
                    value=saved_limits.get("Shopping", 60))
                subs_limit     = gr.Number(
                    label="Subscriptions ($)",
                    value=saved_limits.get("Subscriptions", 30))

            save_budget_btn    = gr.Button("Save Budget Plan",
                                           variant="primary")
            save_budget_status = gr.Textbox(label="Status",
                                            interactive=False)
            with gr.Row():
                back_to_step3 = gr.Button("← Back to Insights",
                                          variant="secondary")
                next_to_step5 = gr.Button("View My Budget →",
                                          variant="primary")

        # ════════════════════════════════════════════════════
        # STEP 5 — YOUR BUDGET
        # ════════════════════════════════════════════════════
        with gr.Group(visible=False) as step5_group:
            gr.HTML('<div class="step-header">Step 5 — Your Budget</div>')

            budget_summary_html = gr.HTML()

            with gr.Group(visible=False) as goals_chart_group:
                gr.HTML('<div style="font-weight:600; font-size:15px; '
                        'color:#1B2A4A; margin:16px 0 8px;">'
                        'Savings Goal Progress</div>')
                goals_chart = gr.Plot(label="Goal Progress")

            gr.HTML("""
<div class="card" style="margin-top:20px;">
  <div style="font-weight:600; font-size:15px; color:#1B2A4A;
              margin-bottom:12px;">Savings Calculator</div>
  <p style="color:#4A5568; font-size:13px; margin-bottom:12px;">
    Calculate how long it will take to reach a specific savings target.
  </p>
</div>""")
            with gr.Row():
                calc_target  = gr.Number(label="Target Amount ($)", value=2000)
                calc_monthly = gr.Number(label="Monthly Contribution ($)", value=150)
                calc_current = gr.Number(label="Already Saved ($)", value=0)
            calc_btn = gr.Button("Calculate", variant="secondary")
            calc_out = gr.Textbox(label="Result", interactive=False, lines=8)

            with gr.Row():
                back_to_step4 = gr.Button("← Back to Plan",
                                          variant="secondary")
                next_to_step6 = gr.Button("Chat with Finn →",
                                          variant="primary")

        # ════════════════════════════════════════════════════
        # STEP 6 — CHAT WITH FINN
        # ════════════════════════════════════════════════════
        with gr.Group(visible=False) as step6_group:
            gr.HTML('<div class="step-header">Step 6 — Chat with Finn</div>')
            gr.HTML("""
<div class="card insight-warning">
  <strong>About Finn:</strong> Finn is a direct financial advisor.
  He will not soften bad news. His advice is based on your actual spending
  data and budget plan. If you are overspending, he will tell you clearly.
</div>""")
            gr.ChatInterface(
                fn       = handle_finn_chat,
                examples = [
                    "Give me an honest summary of my financial situation.",
                    "Where is most of my money going and what should I cut?",
                    "Am I on track to hit my savings goals?",
                    "I keep overspending on nights out — what should I do?",
                    "Are my subscriptions worth keeping?",
                    "What would happen if I missed a week of work?",
                ]
            )

            gr.HTML("""
<div class="card" style="margin-top:24px;">
  <div style="font-weight:600; font-size:15px; color:#1B2A4A;
              margin-bottom:8px;">Upload Next Month</div>
  <p style="color:#4A5568; font-size:13px; margin-bottom:12px;">
    When your next bank statement is ready, upload it here.
    Your budget plan and savings goals will be preserved and the new data
    will be compared against your existing limits automatically.
  </p>
</div>""")
            with gr.Row():
                back_to_step5 = gr.Button("← Back to Budget",
                                          variant="secondary")
                new_month_btn = gr.Button("Upload Next Month's Data →",
                                          variant="primary")

        # ════════════════════════════════════════════════════
        # CLEAR ALL DATA
        # ════════════════════════════════════════════════════
        with gr.Accordion("Reset / Clear All Data", open=False):
            gr.HTML("""
<div class="card insight-critical">
  <strong>Warning:</strong> This will permanently delete all saved data
  including your budget plan, savings goals, and all monthly history.
  This cannot be undone.
</div>""")
            with gr.Row():
                clear_confirm = gr.Textbox(
                    label='Type "YES" to confirm',
                    placeholder="YES",
                    scale=2
                )
                clear_btn = gr.Button("Clear All Data",
                                      variant="stop", scale=1)
            clear_status = gr.Textbox(label="Status", interactive=False)

        # ════════════════════════════════════════════════════
        # EVENT WIRING
        # ════════════════════════════════════════════════════

        # ── Step 1 ────────────────────────────────────────────
        csv_file.change(
            fn      = handle_upload,
            inputs  = [csv_file],
            outputs = [step1_group, step2_group,
                       upload_status, preview_table, nav_bar]
        )
        next_to_step2.click(
            fn      = handle_goto_step2,
            inputs  = [],
            outputs = [step1_group, step2_group, edit_table, nav_bar]
        )

        # ── Step 2 ────────────────────────────────────────────
        back_to_step1.click(
            fn      = lambda: (gr.update(visible=True),
                               gr.update(visible=False),
                               nav_html(1, nav_state["max_reached"])),
            outputs = [step1_group, step2_group, nav_bar]
        )
        save_btn.click(
            fn      = handle_save_edits,
            inputs  = [edit_table],
            outputs = [save_status]
        )
        add_btn.click(
            fn      = handle_add_transaction,
            inputs  = [m_date, m_desc, m_amt, m_cat, m_ess],
            outputs = [add_status, edit_table]
        )
        next_to_step3.click(
            fn      = handle_run_analysis,
            inputs  = [],
            outputs = [step3_group, step2_group,
                       donut_chart, bar_chart, day_chart, gauge_chart,
                       overview_html, insights_html,
                       trend_group, trend_chart,
                       bva_group, bva_html_out, bva_chart,
                       nav_bar]
        )

        # ── Step 3 ────────────────────────────────────────────
        back_to_step2.click(
            fn      = lambda: (gr.update(visible=False),
                               gr.update(visible=True),
                               nav_html(2, nav_state["max_reached"])),
            outputs = [step3_group, step2_group, nav_bar]
        )
        advice_btn.click(
            fn      = handle_generate_advice,
            inputs  = [],
            outputs = [advice_out]
        )

        def next_from_step3_handler():
            returning = app_state.get("budget_plan") is not None
            nav_state["max_reached"] = max(
                nav_state["max_reached"], 5 if returning else 4)
            if returning:
                summary    = build_budget_summary_html(
                    app_state["budget_plan"],
                    app_state["current_analysis"])
                prog_chart = build_savings_progress_chart(
                    app_state["budget_plan"].get("goals", []))
                return (
                    gr.update(visible=False),
                    gr.update(visible=False),
                    gr.update(visible=True),
                    summary,
                    prog_chart if prog_chart else go.Figure(),
                    gr.update(visible=True if prog_chart else False),
                    nav_html(5, nav_state["max_reached"])
                )
            else:
                return (
                    gr.update(visible=False),
                    gr.update(visible=True),
                    gr.update(visible=False),
                    "",
                    go.Figure(),
                    gr.update(visible=False),
                    nav_html(4, nav_state["max_reached"])
                )

        next_from_step3.click(
            fn      = next_from_step3_handler,
            inputs  = [],
            outputs = [step3_group, step4_group, step5_group,
                       budget_summary_html, goals_chart,
                       goals_chart_group, nav_bar]
        )

        # ── Step 4 ────────────────────────────────────────────
        back_to_step3.click(
            fn      = lambda: (gr.update(visible=False),
                               gr.update(visible=True),
                               nav_html(3, nav_state["max_reached"])),
            outputs = [step4_group, step3_group, nav_bar]
        )
        save_budget_btn.click(
            fn      = handle_save_budget,
            inputs  = [monthly_budget, safety_target, safety_current,
                       goal1_name, goal1_target, goal1_current,
                       goal2_name, goal2_target, goal2_current,
                       food_limit, coffee_limit, eating_limit,
                       transport_limit, social_limit, shopping_limit,
                       subs_limit],
            outputs = [save_budget_status, budget_summary_html,
                       goals_chart, goals_chart_group, nav_bar]
        )
        next_to_step5.click(
            fn      = lambda: (gr.update(visible=False),
                               gr.update(visible=True),
                               nav_html(5, nav_state["max_reached"])),
            outputs = [step4_group, step5_group, nav_bar]
        )

        # ── Step 5 ────────────────────────────────────────────
        back_to_step4.click(
            fn      = lambda: (gr.update(visible=False),
                               gr.update(visible=True),
                               nav_html(4, nav_state["max_reached"])),
            outputs = [step5_group, step4_group, nav_bar]
        )
        calc_btn.click(
            fn      = lambda t, m, c: savings_calculator(
                          f"target={t}, monthly={m}, current={c}"),
            inputs  = [calc_target, calc_monthly, calc_current],
            outputs = [calc_out]
        )
        next_to_step6.click(
            fn      = lambda: (gr.update(visible=False),
                               gr.update(visible=True),
                               nav_html(6, nav_state["max_reached"])),
            outputs = [step5_group, step6_group, nav_bar]
        )

        # ── Step 6 ────────────────────────────────────────────
        back_to_step5.click(
            fn      = lambda: (gr.update(visible=False),
                               gr.update(visible=True),
                               nav_html(5, nav_state["max_reached"])),
            outputs = [step6_group, step5_group, nav_bar]
        )
        new_month_btn.click(
            fn      = lambda: (gr.update(visible=True),
                               gr.update(visible=False),
                               None, None,
                               nav_html(1, nav_state["max_reached"])),
            outputs = [step1_group, step6_group,
                       upload_status, preview_table, nav_bar]
        )

        # ── Clear all ─────────────────────────────────────────
        clear_btn.click(
            fn      = handle_clear_all,
            inputs  = [clear_confirm],
            outputs = [clear_status, nav_bar]
        )

    return app


print("Cell 9 ready — Gradio UI defined.")

Cell 9 ready — Gradio UI defined.


In [46]:
# ── Cell 10: Launch ───────────────────────────────────────────

finance_app = create_finance_assistant_ui()
finance_app.launch(debug=True, share=False)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>

Keyboard interruption in main thread... closing server.


---

# 🧪 STEP 6: Test with a Variety of Data

**🔍 Comprehensive Testing Strategy**

Create thorough tests for your Smart Finance Assistant to ensure it handles real-world scenarios.

::: {.callout-tip}
## 🤖 AI Collaboration for Testing

**Effective Testing Prompts:**
```
"Help me create comprehensive test cases for my finance assistant. Include:
- Normal transaction data
- Edge cases (refunds, large amounts, missing data)
- Invalid data scenarios (corrupted files, wrong formats)
- Business logic validation (spending calculations, recommendations)
Create assert statements to verify each scenario."
```
:::

## Foundation Function Tests

In [47]:
# 🤖 AI Collaboration: Comprehensive Test Suite
# Ask AI to help you create thorough test cases

def create_test_datasets():
    """
    Create various test datasets for comprehensive testing

    🤖 AI Collaboration Prompt:
    "Create realistic test datasets for a finance assistant including:
    1. Normal spending data with various categories
    2. Edge cases: refunds (negative amounts), missing data, zero amounts
    3. Data quality issues: invalid formats, extreme values
    4. Business scenarios: high spending months, savings patterns
    Include Australian business names and realistic amounts."
    """
    # Your AI-generated test data goes here
    pass

def test_data_loading_function():
    """
    Test the data loading and cleaning functionality

    🤖 AI Collaboration Prompt:
    "Create assert statements to test my data loading function with:
    - Valid CSV data
    - CSV with dollar signs in amounts
    - Missing values and invalid data
    - Empty files and corrupted data
    Verify that cleaning works correctly and errors are handled gracefully."
    """
    print("🧪 Testing data loading function...")
    # Your AI-generated test cases go here
    pass

def test_spending_analysis():
    """
    Test spending analysis calculations

    🤖 AI Collaboration Prompt:
    "Create tests for spending analysis that verify:
    - Category totals are calculated correctly
    - Percentages add up to 100%
    - Refunds are handled appropriately
    - Edge cases like single transactions or empty categories
    Use assert statements with known expected results."
    """
    print("🧪 Testing spending analysis...")
    # Your AI-generated analysis tests go here
    pass

def test_business_insights():
    """
    Test business recommendation generation

    🤖 AI Collaboration Prompt:
    "Create tests that verify business insights are appropriate:
    - High spending categories are identified correctly
    - Savings opportunities are realistic
    - Recommendations match spending patterns
    - Output format is user-friendly"
    """
    print("🧪 Testing business insights...")
    # Your AI-generated insight tests go here
    pass

# Run all tests
print("🔍 COMPREHENSIVE TESTING SUITE")
print("=" * 40)

try:
    create_test_datasets()
    test_data_loading_function()
    test_spending_analysis()
    test_business_insights()
    print("✅ All tests passed! Your finance assistant is working correctly.")
except AssertionError as e:
    print(f"❌ Test failed: {e}")
except Exception as e:
    print(f"⚠️ Test error: {e}")

🔍 COMPREHENSIVE TESTING SUITE
🧪 Testing data loading function...
🧪 Testing spending analysis...
🧪 Testing business insights...
✅ All tests passed! Your finance assistant is working correctly.


## Advanced Integration Tests

In [48]:
# 🤖 AI Collaboration: Integration Testing
# Ask AI to help test the complete system integration

def test_full_workflow():
    """
    Test the complete workflow from CSV upload to final recommendations

    🤖 AI Collaboration Prompt:
    "Create an end-to-end test that:
    1. Loads sample CSV data
    2. Runs complete analysis pipeline
    3. Generates chat responses about the data
    4. Verifies RAG system retrieval
    5. Tests custom tool functionality
    Ensure all components work together seamlessly."
    """
    print("🧪 Testing complete workflow integration...")
    # Your AI-generated integration tests go here
    pass

def test_error_handling():
    """
    Test error handling and user experience

    🤖 AI Collaboration Prompt:
    "Create tests that verify error handling for:
    - Invalid file uploads
    - Network connection issues
    - Malformed data
    - User input validation
    Ensure error messages are user-friendly and helpful."
    """
    print("🧪 Testing error handling...")
    # Your AI-generated error tests go here
    pass

# Run integration tests
try:
    test_full_workflow()
    test_error_handling()
    print("✅ Integration tests completed successfully!")
except Exception as e:
    print(f"⚠️ Integration test issue: {e}")

🧪 Testing complete workflow integration...
🧪 Testing error handling...
✅ Integration tests completed successfully!


---

# 📊 Project Completion Checklist

## Foundation Skills ✅
- [ ] **Data Processing**: CSV loading and cleaning functions work reliably
- [ ] **Analysis Functions**: Spending summaries calculate correctly
- [ ] **Business Insights**: Recommendations are relevant and actionable  
- [ ] **Error Handling**: Graceful handling of data issues
- [ ] **Testing**: Comprehensive test coverage for core functions
- [ ] **Documentation**: Clear AI collaboration documentation in diary

## Advanced Integration ✅
- [ ] **Chat Interface**: Finance advisor personality implemented
- [ ] **RAG System**: Document retrieval for financial guidance
- [ ] **Custom Tools**: At least one financial calculator/utility
- [ ] **Gradio UI**: Professional, user-friendly interface
- [ ] **Full Integration**: All components work together seamlessly

## Professional Standards ✅
- [ ] **Code Quality**: Professional, commented, maintainable code
- [ ] **Business Focus**: Clear connection to real finance problems
- [ ] **User Experience**: Interface suitable for non-technical users
- [ ] **AI Collaboration**: Extensive, well-documented AI usage
- [ ] **Testing**: Robust validation of all features

## Project Documentation ✅  
- [ ] **Developer's Diary**: Complete AI collaboration documentation
- [ ] **README**: Clear project description and usage instructions
- [ ] **GitHub**: Regular commits showing development progress
- [ ] **Reflection**: Thoughtful analysis of learning and challenges

---

# 🎯 Final Thoughts: Your Finance Assistant Journey

Congratulations on building your Smart Finance Assistant! This project represents a significant achievement in modern business programming:

**Technical Skills Developed:**
- AI-assisted development workflows
- Professional data processing with pandas
- Integration of multiple AI technologies
- User interface design with Gradio
- Comprehensive software testing

**Business Skills Developed:**  
- Financial data analysis and insights
- User-centered application design
- Professional documentation practices
- Iterative development methodology
- Critical evaluation of AI suggestions

**Professional Preparation:**
- Experience with industry-standard AI collaboration
- Portfolio-ready application development
- Understanding of business problem-solving with technology
- Documentation practices for workplace environments

**Your Smart Finance Assistant demonstrates your ability to direct AI tools toward meaningful business solutions - exactly the skill set that modern BIS graduates need for career success!**

---

*Remember to document all AI collaborations in your Developer's Diary and maintain regular GitHub commits throughout your development process.*
